In [8]:
import typing as t
from abc import ABC, abstractmethod
from pathlib import Path
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ICL.eval.collection.data_schema import DataSchemaManager
from ICL.eval.collection.evaluator import create_evaluation_manifest

from pathlib import Path
root_dir = Path("/Users/jliu/workspace/ICL")
data_dir = root_dir / "datasets"
model_dir = root_dir / "models"
result_dir = root_dir / "results"

# Base analysis

In [10]:
"""Revised analysis framework leveraging existing infrastructure."""
class AnalysisSettings:
    """Analysis-specific settings to extend EvaluationConfig."""
    
    def __init__(
        self,
        confidence_level: float = 0.95,
        significance_threshold: float = 0.05,
        n_bootstrap: int = 10000,
        figure_format: str = "png",
        figure_dpi: int = 300,
        figure_size: tuple[int, int] = (10, 8),
        min_samples_per_condition: int = 3,
        max_missing_data_fraction: float = 0.2
    ):
        self.confidence_level = confidence_level
        self.significance_threshold = significance_threshold
        self.n_bootstrap = n_bootstrap
        self.figure_format = figure_format
        self.figure_dpi = figure_dpi
        self.figure_size = figure_size
        self.min_samples_per_condition = min_samples_per_condition
        self.max_missing_data_fraction = max_missing_data_fraction
        
        self._validate()
    
    def _validate(self) -> None:
        """Validate analysis settings."""
        if not (0 < self.confidence_level < 1):
            raise ValueError("Confidence level must be between 0 and 1")
        if not (0 < self.significance_threshold < 1):
            raise ValueError("Significance threshold must be between 0 and 1")


class BaseAnalyzer(ABC):
    """Base class for research question analyzers using existing infrastructure."""
    
    def __init__(
        self, 
        results_dir: Path, 
        analysis_name: str,
        analysis_settings: AnalysisSettings | None = None
    ):
        """Initialize analyzer with existing data infrastructure."""
        self.results_dir = results_dir
        self.analysis_name = analysis_name
        self.settings = analysis_settings or AnalysisSettings()
        
        # Use existing data management infrastructure
        self.data_manager = DataSchemaManager()
        self._setup_output_structure()
        self._setup_visualization()
        
        # Load data using existing patterns
        self._load_evaluation_data()
    
    def _setup_output_structure(self) -> None:
        """Setup output directories following existing patterns."""
        self.output_dir = self.results_dir / "analysis" / self.analysis_name
        
        directories = [
            self.output_dir / "figures",
            self.output_dir / "data", 
            self.output_dir / "reports"
        ]
        
        for directory in directories:
            directory.mkdir(parents=True, exist_ok=True)
    
    def _setup_visualization(self) -> None:
        """Setup visualization parameters."""
        plt.style.use('default')
        sns.set_style("whitegrid")
        sns.set_palette("husl")
        
        plt.rcParams.update({
            'figure.figsize': self.settings.figure_size,
            'figure.dpi': self.settings.figure_dpi,
            'font.size': 12,
            'axes.titlesize': 14,
            'axes.labelsize': 12,
            'xtick.labelsize': 10,
            'ytick.labelsize': 10,
            'legend.fontsize': 11
        })
    
    def _load_evaluation_data(self) -> None:
        """Load evaluation data using existing infrastructure."""
        # Load manifest and registry following existing patterns
        manifest_path = self.results_dir / "metadata" / "experiment_manifest.json"
        registry_path = self.results_dir / "metadata" / "model_registry.parquet"
        performance_path = self.results_dir / "raw_evaluations" / "icl_performance.parquet"
        
        if manifest_path.exists():
            import json
            with open(manifest_path) as f:
                self.manifest = json.load(f)
        else:
            warnings.warn("No experiment manifest found")
            self.manifest = {}
        
        if registry_path.exists():
            self.model_registry = pd.read_parquet(registry_path)
        else:
            warnings.warn("No model registry found")
            self.model_registry = pd.DataFrame()
        
        if performance_path.exists():
            self.performance_data = pd.read_parquet(performance_path)
        else:
            warnings.warn("No performance data found")
            self.performance_data = pd.DataFrame()
    
    @abstractmethod
    def run_analysis(self) -> dict[str, t.Any]:
        """Run the specific analysis for this research question."""
        pass
    
    @abstractmethod
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate analysis report."""
        pass
    
    def save_results(self, results: dict[str, t.Any], filename: str = "results.json") -> Path:
        """Save results using existing serialization patterns."""
        output_path = self.output_dir / "data" / filename
        
        # Use existing serialization approach
        serializable_results = self._convert_to_serializable(results)
        
        import json
        with open(output_path, "w") as f:
            json.dump(serializable_results, f, indent=2, default=str)
        
        return output_path
    
    def save_figure(self, fig: plt.Figure, filename: str) -> Path:
        """Save figure following existing patterns."""
        output_path = self.output_dir / "figures" / f"{filename}.{self.settings.figure_format}"
        fig.tight_layout()
        fig.savefig(output_path, dpi=self.settings.figure_dpi, bbox_inches='tight')
        plt.close(fig)
        return output_path
    
    def validate_data_completeness(self, data: pd.DataFrame, required_columns: list[str]) -> bool:
        """Validate data completeness using existing patterns."""
        if data.empty:
            warnings.warn("No data available for analysis")
            return False
        
        # Check required columns
        missing_columns = [col for col in required_columns if col not in data.columns]
        if missing_columns:
            warnings.warn(f"Missing required columns: {missing_columns}")
            return False
        
        # Check missing data threshold
        for col in required_columns:
            missing_fraction = data[col].isna().mean()
            if missing_fraction > self.settings.max_missing_data_fraction:
                warnings.warn(f"Column {col} has {missing_fraction:.1%} missing data")
                return False
        
        # Check minimum sample size
        if len(data) < self.settings.min_samples_per_condition:
            warnings.warn(f"Insufficient data: {len(data)} < {self.settings.min_samples_per_condition}")
            return False
        
        return True
    
    def create_summary_statistics(self, data: pd.DataFrame, group_by: list[str]) -> pd.DataFrame:
        """Create summary statistics."""
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        
        summary = data.groupby(group_by)[numeric_cols].agg([
            'count', 'mean', 'std', 'min', 'max', 'median'
        ]).round(4)
        
        return summary
    
    def _convert_to_serializable(self, obj: t.Any) -> t.Any:
        """Convert to JSON serializable format using existing patterns."""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        elif isinstance(obj, np.bool_):
            return bool(obj)
        elif isinstance(obj, dict):
            return {key: self._convert_to_serializable(value) for key, value in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [self._convert_to_serializable(item) for item in obj]
        elif isinstance(obj, Path):
            return str(obj)
        elif pd.isna(obj):
            return None
        else:
            return obj

In [13]:
perf_path = result_dir/"raw/metadata/model_registry.parquet"
data = pd.read_parquet(perf_path)

In [14]:
data

,model_id,config_L,config_m,n_train,checkpoint_step,checkpoint_path,model_type,training_seed,eval_seed
0,causal_lm_L2_m3_ntrain64_step1000_233039,2,3,64,1000,/Users/jliu/workspace/ICL/models/rhm_clm_train...,causal_lm,<NA>,<NA>
1,causal_lm_L2_m3_ntrain64_step1500_766766,2,3,64,1500,/Users/jliu/workspace/ICL/models/rhm_clm_train...,causal_lm,<NA>,<NA>
2,causal_lm_L2_m3_ntrain64_step500_884450,2,3,64,500,/Users/jliu/workspace/ICL/models/rhm_clm_train...,causal_lm,<NA>,<NA>


# Analysis util

In [11]:
"""Statistical analysis utilities extracted for reuse across analyzers."""

import typing as t
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score


@dataclass
class StatisticalResult:
    """Standardized result for statistical tests."""
    success: bool
    test_name: str
    statistic: float | None = None
    p_value: float | None = None
    significant: bool | None = None
    effect_size: float | None = None
    group1_stats: dict[str, float] | None = None
    group2_stats: dict[str, float] | None = None
    error: str | None = None


@dataclass
class CurveFitResult:
    """Standardized result for curve fitting."""
    success: bool
    curve_type: str
    parameters: dict[str, float] | None = None
    parameter_errors: dict[str, float] | None = None
    r2: float | None = None
    predictions: list[float] | None = None
    equation: str | None = None
    error: str | None = None


class StatisticalAnalyzer:
    """Statistical analysis utilities."""
    
    def __init__(self, significance_threshold: float = 0.05, n_bootstrap: int = 10000):
        self.significance_threshold = significance_threshold
        self.n_bootstrap = n_bootstrap
    
    def bootstrap_confidence_interval(
        self, 
        data: list[float], 
        confidence_level: float = 0.95,
        statistic_func: t.Callable = np.mean
    ) -> tuple[float, float]:
        """Compute bootstrap confidence interval."""
        if len(data) < 2:
            return float('nan'), float('nan')
        
        bootstrap_stats = []
        for _ in range(self.n_bootstrap):
            sample = np.random.choice(data, size=len(data), replace=True)
            bootstrap_stats.append(statistic_func(sample))
        
        alpha = 1 - confidence_level
        lower = np.percentile(bootstrap_stats, 100 * alpha / 2)
        upper = np.percentile(bootstrap_stats, 100 * (1 - alpha / 2))
        
        return float(lower), float(upper)
    
    def compare_groups(
        self, 
        group1: list[float], 
        group2: list[float], 
        test_type: str = "ttest"
    ) -> StatisticalResult:
        """Perform statistical test between two groups."""
        if len(group1) < 2 or len(group2) < 2:
            return StatisticalResult(
                success=False,
                test_name=test_type,
                error="Insufficient data for statistical test"
            )
        
        try:
            if test_type == "ttest":
                statistic, p_value = stats.ttest_ind(group1, group2)
                test_name = "Independent t-test"
            elif test_type == "mannwhitney":
                statistic, p_value = stats.mannwhitneyu(group1, group2, alternative='two-sided')
                test_name = "Mann-Whitney U test"
            elif test_type == "ks":
                statistic, p_value = stats.ks_2samp(group1, group2)
                test_name = "Kolmogorov-Smirnov test"
            else:
                return StatisticalResult(
                    success=False,
                    test_name=test_type,
                    error=f"Unknown test type: {test_type}"
                )
            
            effect_size = self._calculate_cohens_d(group1, group2)
            significant = p_value < self.significance_threshold
            
            return StatisticalResult(
                success=True,
                test_name=test_name,
                statistic=float(statistic),
                p_value=float(p_value),
                significant=significant,
                effect_size=effect_size,
                group1_stats={"mean": np.mean(group1), "std": np.std(group1), "n": len(group1)},
                group2_stats={"mean": np.mean(group2), "std": np.std(group2), "n": len(group2)}
            )
            
        except Exception as e:
            return StatisticalResult(
                success=False,
                test_name=test_type,
                error=f"Statistical test failed: {e}"
            )
    
    def _calculate_cohens_d(self, group1: list[float], group2: list[float]) -> float:
        """Calculate Cohen's d effect size."""
        mean1, mean2 = np.mean(group1), np.mean(group2)
        std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
        n1, n2 = len(group1), len(group2)
        
        # Pooled standard deviation
        pooled_std = np.sqrt(((n1 - 1) * std1**2 + (n2 - 1) * std2**2) / (n1 + n2 - 2))
        
        if pooled_std == 0:
            return 0.0
        
        return (mean1 - mean2) / pooled_std


class CurveFitter:
    """Curve fitting utilities with standardized interface."""
    
    # Define curve functions as class constants
    CURVE_FUNCTIONS = {
        "logistic": lambda x, L, k, x0: L / (1 + np.exp(-k * (x - x0))),
        "exponential": lambda x, a, b: a * np.exp(b * x),
        "power": lambda x, a, b: a * np.power(x, b)
    }
    
    def fit_curve(
        self, 
        x: np.ndarray, 
        y: np.ndarray, 
        curve_type: str = "logistic"
    ) -> CurveFitResult:
        """Fit curve to data with unified interface."""
        if len(x) < 3 or len(y) < 3:
            return CurveFitResult(
                success=False,
                curve_type=curve_type,
                error="Insufficient data points"
            )
        
        if curve_type == "polynomial":
            return self._fit_polynomial(x, y)
        elif curve_type in self.CURVE_FUNCTIONS:
            return self._fit_parametric_curve(x, y, curve_type)
        else:
            return CurveFitResult(
                success=False,
                curve_type=curve_type,
                error=f"Unknown curve type: {curve_type}"
            )
    
    def _fit_parametric_curve(self, x: np.ndarray, y: np.ndarray, curve_type: str) -> CurveFitResult:
        """Fit parametric curves using unified approach."""
        func = self.CURVE_FUNCTIONS[curve_type]
        
        try:
            # Get initial parameter estimates
            initial_params = self._get_initial_params(x, y, curve_type)
            
            # Fit curve
            params, covariance = curve_fit(func, x, y, p0=initial_params, maxfev=5000)
            
            # Calculate predictions and R²
            y_pred = func(x, *params)
            r2 = r2_score(y, y_pred)
            
            # Calculate parameter uncertainties
            param_errors = np.sqrt(np.diag(covariance))
            
            # Create parameter dictionaries
            param_names = self._get_param_names(curve_type)
            parameters = dict(zip(param_names, params))
            parameter_errors = dict(zip(param_names, param_errors))
            
            # Generate equation string
            equation = self._generate_equation(curve_type, parameters)
            
            return CurveFitResult(
                success=True,
                curve_type=curve_type,
                parameters=parameters,
                parameter_errors=parameter_errors,
                r2=r2,
                predictions=y_pred.tolist(),
                equation=equation
            )
            
        except Exception as e:
            return CurveFitResult(
                success=False,
                curve_type=curve_type,
                error=f"Curve fitting failed: {e}"
            )
    
    def _fit_polynomial(self, x: np.ndarray, y: np.ndarray, degree: int = 2) -> CurveFitResult:
        """Fit polynomial curve."""
        try:
            coeffs = np.polyfit(x, y, degree)
            y_pred = np.polyval(coeffs, x)
            r2 = r2_score(y, y_pred)
            
            # Create equation string
            terms = []
            for i, coeff in enumerate(coeffs):
                power = degree - i
                if power == 0:
                    terms.append(f"{coeff:.3f}")
                elif power == 1:
                    terms.append(f"{coeff:.3f}*x")
                else:
                    terms.append(f"{coeff:.3f}*x^{power}")
            
            equation = "y = " + " + ".join(terms)
            
            return CurveFitResult(
                success=True,
                curve_type=f"polynomial_degree_{degree}",
                parameters={"coefficients": coeffs.tolist(), "degree": degree},
                r2=r2,
                predictions=y_pred.tolist(),
                equation=equation
            )
            
        except Exception as e:
            return CurveFitResult(
                success=False,
                curve_type=f"polynomial_degree_{degree}",
                error=f"Polynomial fit failed: {e}"
            )
    
    def _get_initial_params(self, x: np.ndarray, y: np.ndarray, curve_type: str) -> list[float]:
        """Get initial parameter estimates for curve fitting."""
        if curve_type == "logistic":
            L_init = np.max(y)
            k_init = 1.0
            x0_init = np.median(x)
            return [L_init, k_init, x0_init]
        
        elif curve_type == "exponential":
            # Use log transform for initial guess
            log_y = np.log(np.maximum(y, 1e-10))
            poly_fit = np.polyfit(x, log_y, 1)
            a_init = np.exp(poly_fit[1])
            b_init = poly_fit[0]
            return [a_init, b_init]
        
        elif curve_type == "power":
            # Use log transform for initial guess
            mask = (x > 0) & (y > 0)
            if np.sum(mask) < 3:
                return [1.0, 1.0]  # Default guess
            
            x_pos, y_pos = x[mask], y[mask]
            log_x, log_y = np.log(x_pos), np.log(y_pos)
            poly_fit = np.polyfit(log_x, log_y, 1)
            a_init = np.exp(poly_fit[1])
            b_init = poly_fit[0]
            return [a_init, b_init]
        
        else:
            return [1.0, 1.0]  # Default guess
    
    def _get_param_names(self, curve_type: str) -> list[str]:
        """Get parameter names for each curve type."""
        param_names = {
            "logistic": ["L", "k", "x0"],
            "exponential": ["a", "b"],
            "power": ["a", "b"]
        }
        return param_names.get(curve_type, ["param1", "param2"])
    
    def _generate_equation(self, curve_type: str, parameters: dict[str, float]) -> str:
        """Generate equation string for the fitted curve."""
        if curve_type == "logistic":
            L, k, x0 = parameters["L"], parameters["k"], parameters["x0"]
            return f"y = {L:.3f} / (1 + exp(-{k:.3f} * (x - {x0:.3f})))"
        
        elif curve_type == "exponential":
            a, b = parameters["a"], parameters["b"]
            return f"y = {a:.3f} * exp({b:.3f} * x)"
        
        elif curve_type == "power":
            a, b = parameters["a"], parameters["b"]
            return f"y = {a:.3f} * x^{b:.3f}"
        
        else:
            return "Equation format not defined"


class VisualizationManager:
    """Manages consistent visualization across analyzers."""
    
    def __init__(self, style: str = "whitegrid", palette: str = "husl"):
        self.style = style
        self.palette = palette
        self._setup_style()
    
    def _setup_style(self) -> None:
        """Setup consistent styling."""
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        plt.style.use('default')
        sns.set_style(self.style)
        sns.set_palette(self.palette)
    
    def create_comparison_plot(
        self, 
        data: pd.DataFrame, 
        x: str, 
        y: str, 
        hue: str | None = None,
        plot_type: str = "line"
    ) -> plt.Figure:
        """Create standardized comparison plots."""
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        if plot_type == "line":
            sns.lineplot(data=data, x=x, y=y, hue=hue, ax=ax, marker='o')
        elif plot_type == "bar":
            sns.barplot(data=data, x=x, y=y, hue=hue, ax=ax)
        elif plot_type == "box":
            sns.boxplot(data=data, x=x, y=y, hue=hue, ax=ax)
        elif plot_type == "scatter":
            sns.scatterplot(data=data, x=x, y=y, hue=hue, ax=ax)
        
        ax.set_title(f"{y} vs {x}")
        return fig
    
    def create_emergence_plot(
        self, 
        data: pd.DataFrame, 
        x: str = "context_size", 
        y: str = "accuracy"
    ) -> plt.Figure:
        """Create emergence-specific visualization."""
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Plot emergence curves
        sns.lineplot(
            data=data, 
            x=x, 
            y=y, 
            hue="config_L",
            style="config_m",
            markers=True,
            ax=ax
        )
        
        # Add emergence threshold line
        ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Emergence threshold')
        
        ax.set_title("ICL Emergence Analysis")
        ax.set_xlabel("Context Size (k)")
        ax.set_ylabel("Accuracy")
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        return fig

# RQ1: Emergence Analysis

In [ ]:
"""Example RQ1 Emergence Analyzer using the revised framework."""
class RQ1EmergenceAnalyzer(BaseAnalyzer):
    """Research Question 1: Emergence Analysis using revised framework."""
    
    def __init__(self, results_dir: Path, analysis_settings: AnalysisSettings | None = None):
        """Initialize emergence analyzer."""
        super().__init__(results_dir, "rq1_emergence", analysis_settings)
        
        # Initialize analysis utilities
        self.stats = StatisticalAnalyzer(self.settings.significance_threshold)
        self.curve_fitter = CurveFitter()
        self.viz = VisualizationManager()
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run emergence analysis."""
        # Validate data completeness
        required_columns = ["context_size", "accuracy", "config_L", "config_m", "control_type"]
        if not self.validate_data_completeness(self.performance_data, required_columns):
            raise ValueError("Insufficient data for emergence analysis")
        
        # Filter for normal sequences only
        normal_data = self.performance_data[
            self.performance_data["control_type"] == "normal"
        ].copy()
        
        # Analyze emergence patterns
        emergence_results = self._analyze_emergence_patterns(normal_data)
        
        # Fit emergence curves
        curve_results = self._fit_emergence_curves(normal_data)
        
        # Statistical analysis
        statistical_results = self._statistical_analysis(normal_data)
        
        # Generate visualizations
        figures = self._create_visualizations(normal_data, emergence_results)
        
        return {
            "emergence_analysis": emergence_results,
            "curve_fitting": curve_results,
            "statistical_tests": statistical_results,
            "figures": figures,
            "summary": self._create_summary(emergence_results, curve_results)
        }
    
    def _analyze_emergence_patterns(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze emergence patterns across configurations."""
        emergence_results = {}
        
        # Group by configuration
        for (config_L, config_m), group in data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Calculate emergence threshold (minimum k for >50% accuracy)
            emergence_threshold = self._calculate_emergence_threshold(group)
            
            # Calculate max accuracy achieved
            max_accuracy = group["accuracy"].max()
            
            # Calculate accuracy progression
            context_progression = group.groupby("context_size")["accuracy"].mean().to_dict()
            
            emergence_results[config_key] = {
                "emergence_threshold": emergence_threshold,
                "max_accuracy": max_accuracy,
                "context_progression": context_progression,
                "sample_size": len(group)
            }
        
        return emergence_results
    
    def _calculate_emergence_threshold(self, group: pd.DataFrame) -> float:
        """Calculate emergence threshold following existing logic."""
        threshold = 0.5
        
        # Group by context size and compute mean accuracy
        by_context = group.groupby("context_size")["accuracy"].mean().sort_index()
        
        # Find first context size with accuracy > threshold
        for context_size, accuracy in by_context.items():
            if accuracy > threshold:
                return float(context_size)
        
        # If no emergence, return max context size + 1
        return float(max(group["context_size"]) + 1) if len(group) > 0 else float("inf")
    
    def _fit_emergence_curves(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Fit emergence curves for each configuration."""
        curve_results = {}
        
        for (config_L, config_m), group in data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Prepare data for curve fitting
            context_accuracy = group.groupby("context_size")["accuracy"].mean()
            x = np.array(context_accuracy.index)
            y = np.array(context_accuracy.values)
            
            if len(x) < 3:
                curve_results[config_key] = {"error": "Insufficient data for curve fitting"}
                continue
            
            # Try different curve types
            curve_types = ["logistic", "exponential", "polynomial"]
            best_fit = None
            best_r2 = -1
            
            for curve_type in curve_types:
                result = self.curve_fitter.fit_curve(x, y, curve_type)
                if result.success and result.r2 and result.r2 > best_r2:
                    best_r2 = result.r2
                    best_fit = result
            
            curve_results[config_key] = best_fit.__dict__ if best_fit else {"error": "No successful fits"}
        
        return curve_results
    
    def _statistical_analysis(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Perform statistical tests on emergence patterns."""
        statistical_results = {}
        
        # Compare emergence across different L values
        l_values = sorted(data["config_L"].unique())
        if len(l_values) >= 2:
            for i, l1 in enumerate(l_values[:-1]):
                for l2 in l_values[i+1:]:
                    group1_data = data[data["config_L"] == l1]
                    group2_data = data[data["config_L"] == l2]
                    
                    # Calculate emergence thresholds for comparison
                    thresholds1 = []
                    thresholds2 = []
                    
                    for config_m in data["config_m"].unique():
                        g1 = group1_data[group1_data["config_m"] == config_m]
                        g2 = group2_data[group2_data["config_m"] == config_m]
                        
                        if len(g1) > 0:
                            thresholds1.append(self._calculate_emergence_threshold(g1))
                        if len(g2) > 0:
                            thresholds2.append(self._calculate_emergence_threshold(g2))
                    
                    if len(thresholds1) >= 2 and len(thresholds2) >= 2:
                        test_result = self.stats.compare_groups(thresholds1, thresholds2, "ttest")
                        statistical_results[f"L{l1}_vs_L{l2}_emergence"] = test_result.__dict__
        
        # Compare emergence across different m values
        m_values = sorted(data["config_m"].unique())
        if len(m_values) >= 2:
            for i, m1 in enumerate(m_values[:-1]):
                for m2 in m_values[i+1:]:
                    group1_data = data[data["config_m"] == m1]
                    group2_data = data[data["config_m"] == m2]
                    
                    # Calculate emergence thresholds for comparison
                    thresholds1 = []
                    thresholds2 = []
                    
                    for config_L in data["config_L"].unique():
                        g1 = group1_data[group1_data["config_L"] == config_L]
                        g2 = group2_data[group2_data["config_L"] == config_L]
                        
                        if len(g1) > 0:
                            thresholds1.append(self._calculate_emergence_threshold(g1))
                        if len(g2) > 0:
                            thresholds2.append(self._calculate_emergence_threshold(g2))
                    
                    if len(thresholds1) >= 2 and len(thresholds2) >= 2:
                        test_result = self.stats.compare_groups(thresholds1, thresholds2, "ttest")
                        statistical_results[f"m{m1}_vs_m{m2}_emergence"] = test_result.__dict__
        
        return statistical_results
    
    def _create_visualizations(self, data: pd.DataFrame, emergence_results: dict) -> dict[str, str]:
        """Create emergence analysis visualizations."""
        figures = {}
        
        # 1. Main emergence plot
        fig = self.viz.create_emergence_plot(data)
        figures["emergence_curves"] = str(self.save_figure(fig, "emergence_curves"))
        
        # 2. Emergence threshold heatmap
        fig = self._create_threshold_heatmap(emergence_results)
        figures["threshold_heatmap"] = str(self.save_figure(fig, "threshold_heatmap"))
        
        # 3. Configuration comparison
        fig = self._create_config_comparison(data)
        figures["config_comparison"] = str(self.save_figure(fig, "config_comparison"))
        
        # 4. Context size progression
        fig = self._create_progression_plot(data)
        figures["progression_plot"] = str(self.save_figure(fig, "progression_plot"))
        
        return figures
    
    def _create_threshold_heatmap(self, emergence_results: dict) -> plt.Figure:
        """Create heatmap of emergence thresholds."""
        import seaborn as sns
        
        # Extract L and m values and thresholds
        config_data = []
        for config_key, results in emergence_results.items():
            if "emergence_threshold" in results:
                # Parse L and m from config_key (format: "L2_m3")
                parts = config_key.split("_")
                L = int(parts[0][1:])  # Remove 'L' prefix
                m = int(parts[1][1:])  # Remove 'm' prefix
                threshold = results["emergence_threshold"]
                
                config_data.append({"L": L, "m": m, "threshold": threshold})
        
        if not config_data:
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.text(0.5, 0.5, "No emergence data available", ha='center', va='center')
            return fig
        
        # Create pivot table for heatmap
        df_heatmap = pd.DataFrame(config_data).pivot(index="L", columns="m", values="threshold")
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(
            df_heatmap, 
            annot=True, 
            fmt=".1f", 
            cmap="RdYlBu_r",
            ax=ax,
            cbar_kws={'label': 'Emergence Threshold (k)'}
        )
        ax.set_title("Emergence Threshold by Configuration")
        ax.set_xlabel("Number of Features (m)")
        ax.set_ylabel("Sequence Length (L)")
        
        return fig
    
    def _create_config_comparison(self, data: pd.DataFrame) -> plt.Figure:
        """Create configuration comparison plot."""
        import seaborn as sns
        
        # Calculate mean accuracy by configuration and context size
        config_accuracy = data.groupby(["config_L", "config_m", "context_size"])["accuracy"].mean().reset_index()
        config_accuracy["config"] = config_accuracy["config_L"].astype(str) + "_" + config_accuracy["config_m"].astype(str)
        
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot by L
        sns.lineplot(
            data=config_accuracy,
            x="context_size",
            y="accuracy", 
            hue="config_L",
            ax=axes[0],
            marker='o'
        )
        axes[0].set_title("Emergence by Sequence Length (L)")
        axes[0].axhline(y=0.5, color='red', linestyle='--', alpha=0.7)
        axes[0].set_xlabel("Context Size (k)")
        axes[0].set_ylabel("Accuracy")
        
        # Plot by m
        sns.lineplot(
            data=config_accuracy,
            x="context_size",
            y="accuracy",
            hue="config_m", 
            ax=axes[1],
            marker='s'
        )
        axes[1].set_title("Emergence by Number of Features (m)")
        axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.7)
        axes[1].set_xlabel("Context Size (k)")
        axes[1].set_ylabel("Accuracy")
        
        plt.tight_layout()
        return fig
    
    def _create_progression_plot(self, data: pd.DataFrame) -> plt.Figure:
        """Create detailed progression plot with confidence intervals."""
        import seaborn as sns
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Plot with confidence intervals
        sns.lineplot(
            data=data,
            x="context_size",
            y="accuracy",
            hue="config_L",
            style="config_m",
            markers=True,
            err_style="band",
            ax=ax
        )
        
        # Add emergence threshold
        ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Emergence threshold (50%)')
        
        # Customize plot
        ax.set_title("ICL Emergence Progression with Confidence Intervals")
        ax.set_xlabel("Context Size (k)")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        
        return fig
    
    def _create_summary(self, emergence_results: dict, curve_results: dict) -> dict[str, t.Any]:
        """Create analysis summary."""
        # Calculate overall statistics
        thresholds = [
            r["emergence_threshold"] for r in emergence_results.values() 
            if "emergence_threshold" in r and r["emergence_threshold"] != float("inf")
        ]
        
        max_accuracies = [
            r["max_accuracy"] for r in emergence_results.values()
            if "max_accuracy" in r
        ]
        
        successful_fits = sum(
            1 for r in curve_results.values() 
            if isinstance(r, dict) and r.get("success", False)
        )
        
        summary = {
            "total_configurations": len(emergence_results),
            "configurations_with_emergence": len(thresholds),
            "mean_emergence_threshold": np.mean(thresholds) if thresholds else None,
            "std_emergence_threshold": np.std(thresholds) if thresholds else None,
            "min_emergence_threshold": min(thresholds) if thresholds else None,
            "max_emergence_threshold": max(thresholds) if thresholds else None,
            "mean_max_accuracy": np.mean(max_accuracies) if max_accuracies else None,
            "successful_curve_fits": successful_fits,
            "curve_fit_success_rate": successful_fits / len(curve_results) if curve_results else 0
        }
        
        return summary
    
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate comprehensive emergence analysis report."""
        report_path = self.config.output_dir / "reports" / "rq1_emergence_report.md"
        
        with open(report_path, "w") as f:
            f.write("# RQ1: ICL Emergence Analysis Report\n\n")
            f.write(f"Generated: {pd.Timestamp.now()}\n\n")
            
            # Executive Summary
            f.write("## Executive Summary\n\n")
            summary = results["summary"]
            f.write(f"- **Total Configurations Analyzed**: {summary['total_configurations']}\n")
            f.write(f"- **Configurations Showing Emergence**: {summary['configurations_with_emergence']}\n")
            
            if summary["mean_emergence_threshold"]:
                f.write(f"- **Mean Emergence Threshold**: {summary['mean_emergence_threshold']:.2f} ± {summary['std_emergence_threshold']:.2f}\n")
                f.write(f"- **Emergence Threshold Range**: {summary['min_emergence_threshold']:.1f} - {summary['max_emergence_threshold']:.1f}\n")
            
            if summary["mean_max_accuracy"]:
                f.write(f"- **Mean Maximum Accuracy**: {summary['mean_max_accuracy']:.3f}\n")
            
            f.write(f"- **Curve Fitting Success Rate**: {summary['curve_fit_success_rate']:.1%}\n\n")
            
            # Detailed Results
            f.write("## Emergence Analysis by Configuration\n\n")
            for config_key, config_results in results["emergence_analysis"].items():
                f.write(f"### Configuration {config_key}\n")
                f.write(f"- Emergence Threshold: {config_results['emergence_threshold']:.2f}\n")
                f.write(f"- Maximum Accuracy: {config_results['max_accuracy']:.3f}\n")
                f.write(f"- Sample Size: {config_results['sample_size']}\n\n")
            
            # Statistical Tests
            f.write("## Statistical Analysis\n\n")
            for test_name, test_result in results["statistical_tests"].items():
                if test_result.get("success"):
                    f.write(f"### {test_name.replace('_', ' ').title()}\n")
                    f.write(f"- Test: {test_result['test_name']}\n")
                    f.write(f"- p-value: {test_result['p_value']:.4f}\n")
                    f.write(f"- Significant: {'Yes' if test_result['significant'] else 'No'}\n")
                    f.write(f"- Effect Size (Cohen's d): {test_result['effect_size']:.3f}\n\n")
            
            # Curve Fitting Results
            f.write("## Curve Fitting Analysis\n\n")
            for config_key, curve_result in results["curve_fitting"].items():
                if curve_result.get("success"):
                    f.write(f"### {config_key}\n")
                    f.write(f"- Best Fit: {curve_result['curve_type']}\n")
                    f.write(f"- R²: {curve_result['r2']:.4f}\n")
                    f.write(f"- Equation: {curve_result['equation']}\n\n")
            
            # Figures
            f.write("## Generated Figures\n\n")
            for fig_name, fig_path in results["figures"].items():
                f.write(f"- **{fig_name.replace('_', ' ').title()}**: `{fig_path}`\n")
        
        return report_path


# Example usage function
def run_emergence_analysis(
    results_dir: str | Path,
    output_dir: str | Path | None = None,
    **analysis_kwargs
) -> dict[str, t.Any]:
    """Run emergence analysis with custom settings."""
    results_path = Path(results_dir)
    
    # Setup analysis settings
    settings = AnalysisSettings(**analysis_kwargs)
    
    # Initialize and run analyzer
    analyzer = RQ1EmergenceAnalyzer(results_path, settings)
    results = analyzer.run_analysis()
    
    # Save results and generate report
    analyzer.save_results(results)
    report_path = analyzer.generate_report(results)
    
    print(f"Emergence analysis completed!")
    print(f"Results saved to: {analyzer.output_dir}")
    print(f"Report: {report_path}")
    
    return results

In [12]:
# Simple usage
results_dir=result_dir/"raw"
results = run_emergence_analysis(
    results_dir=results_dir,
    confidence_level=0.95,
    n_bootstrap=10000
)

# Advanced usage with custom settings
settings = AnalysisSettings(
    figure_dpi=300,
    significance_threshold=0.01
)
analyzer = RQ1EmergenceAnalyzer(results_dir, settings)
results = analyzer.run_analysis()

/var/folders/c2/336gdlh133qfynjgt7nm9jhw0000gn/T/ipykernel_95509/3428398014.py:126: UserWarning: No performance data found
  warnings.warn("No performance data found")
/var/folders/c2/336gdlh133qfynjgt7nm9jhw0000gn/T/ipykernel_95509/3428398014.py:163: UserWarning: No data available for analysis
  warnings.warn("No data available for analysis")


ValueError: Insufficient data for emergence analysis

# RQ2: Scaling Law analyis

In [ ]:
"""RQ2: Scaling Laws Analysis - How ICL performance scales with model parameters."""

import typing as t
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from .base_analyzer import BaseAnalyzer, AnalysisSettings
from .analysis_utils import StatisticalAnalyzer, CurveFitter, VisualizationManager


class RQ2ScalingAnalyzer(BaseAnalyzer):
    """Research Question 2: Scaling Laws Analysis."""
    
    def __init__(self, results_dir: Path, analysis_settings: AnalysisSettings | None = None):
        """Initialize scaling laws analyzer."""
        super().__init__(results_dir, "rq2_scaling", analysis_settings)
        
        # Initialize analysis utilities
        self.stats = StatisticalAnalyzer(self.settings.significance_threshold)
        self.curve_fitter = CurveFitter()
        self.viz = VisualizationManager()
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run scaling laws analysis."""
        # Validate data
        required_columns = ["config_L", "config_m", "n_train", "context_size", "accuracy", "control_type"]
        if not self.validate_data_completeness(self.performance_data, required_columns):
            raise ValueError("Insufficient data for scaling analysis")
        
        # Filter for normal sequences only
        normal_data = self.performance_data[
            self.performance_data["control_type"] == "normal"
        ].copy()
        
        # Calculate model parameters
        normal_data = self._calculate_model_parameters(normal_data)
        
        # Analyze scaling relationships
        scaling_results = self._analyze_scaling_relationships(normal_data)
        
        # Fit scaling laws
        scaling_laws = self._fit_scaling_laws(normal_data)
        
        # Analyze parameter interactions
        interaction_results = self._analyze_parameter_interactions(normal_data)
        
        # Generate visualizations
        figures = self._create_visualizations(normal_data, scaling_results)
        
        return {
            "scaling_analysis": scaling_results,
            "scaling_laws": scaling_laws,
            "parameter_interactions": interaction_results,
            "figures": figures,
            "summary": self._create_summary(scaling_results, scaling_laws)
        }
    
    def _calculate_model_parameters(self, data: pd.DataFrame) -> pd.DataFrame:
        """Calculate model parameters for scaling analysis."""
        # Calculate total parameters (simplified transformer parameter count)
        # Approximate: P ≈ L × d² × 12 (where d is model dimension)
        # For our analysis, use L × m as proxy for model complexity
        data["model_params"] = data["config_L"] * data["config_m"]
        
        # Calculate effective training data per parameter
        data["data_per_param"] = data["n_train"] / data["model_params"]
        
        # Log scales for power law analysis
        data["log_params"] = np.log10(data["model_params"])
        data["log_n_train"] = np.log10(data["n_train"])
        data["log_context_size"] = np.log10(data["context_size"])
        
        return data
    
    def _analyze_scaling_relationships(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze scaling relationships across different dimensions."""
        scaling_results = {}
        
        # 1. Parameter count scaling (at fixed context size and training data)
        param_scaling = self._analyze_parameter_scaling(data)
        scaling_results["parameter_scaling"] = param_scaling
        
        # 2. Training data scaling (at fixed parameters and context size)
        data_scaling = self._analyze_training_data_scaling(data)
        scaling_results["training_data_scaling"] = data_scaling
        
        # 3. Context size scaling (at fixed parameters and training data)
        context_scaling = self._analyze_context_scaling(data)
        scaling_results["context_scaling"] = context_scaling
        
        # 4. Combined scaling analysis
        combined_scaling = self._analyze_combined_scaling(data)
        scaling_results["combined_scaling"] = combined_scaling
        
        return scaling_results
    
    def _analyze_parameter_scaling(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how performance scales with model parameters."""
        param_results = {}
        
        # Group by fixed context size and training data
        for context_size in sorted(data["context_size"].unique()):
            for n_train in sorted(data["n_train"].unique()):
                subset = data[
                    (data["context_size"] == context_size) & 
                    (data["n_train"] == n_train)
                ]
                
                if len(subset) < 3:
                    continue
                
                # Calculate mean accuracy by parameter count
                param_acc = subset.groupby("model_params")["accuracy"].agg(["mean", "std", "count"]).reset_index()
                
                if len(param_acc) < 3:
                    continue
                
                key = f"k{context_size}_n{n_train}"
                
                # Fit power law: accuracy = a * params^b
                x = param_acc["model_params"].values
                y = param_acc["mean"].values
                
                power_fit = self.curve_fitter.fit_curve(x, y, "power")
                
                # Calculate correlation with log scale
                log_corr = np.corrcoef(np.log10(x), y)[0, 1] if len(x) > 1 else 0
                
                param_results[key] = {
                    "data_points": len(param_acc),
                    "parameter_range": [int(x.min()), int(x.max())],
                    "accuracy_range": [float(y.min()), float(y.max())],
                    "power_law_fit": power_fit.__dict__,
                    "log_correlation": float(log_corr),
                    "raw_data": param_acc.to_dict("records")
                }
        
        return param_results
    
    def _analyze_training_data_scaling(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how performance scales with training data."""
        data_results = {}
        
        # Group by fixed parameters and context size
        for (config_L, config_m) in data[["config_L", "config_m"]].drop_duplicates().values:
            for context_size in sorted(data["context_size"].unique()):
                subset = data[
                    (data["config_L"] == config_L) & 
                    (data["config_m"] == config_m) & 
                    (data["context_size"] == context_size)
                ]
                
                if len(subset) < 3:
                    continue
                
                # Calculate mean accuracy by training data
                data_acc = subset.groupby("n_train")["accuracy"].agg(["mean", "std", "count"]).reset_index()
                
                if len(data_acc) < 3:
                    continue
                
                key = f"L{config_L}_m{config_m}_k{context_size}"
                
                # Fit power law and log curve
                x = data_acc["n_train"].values
                y = data_acc["mean"].values
                
                power_fit = self.curve_fitter.fit_curve(x, y, "power")
                log_fit = self.curve_fitter.fit_curve(np.log10(x), y, "polynomial")
                
                data_results[key] = {
                    "data_points": len(data_acc),
                    "n_train_range": [int(x.min()), int(x.max())],
                    "accuracy_range": [float(y.min()), float(y.max())],
                    "power_law_fit": power_fit.__dict__,
                    "log_fit": log_fit.__dict__,
                    "raw_data": data_acc.to_dict("records")
                }
        
        return data_results
    
    def _analyze_context_scaling(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how performance scales with context size."""
        context_results = {}
        
        # Group by fixed parameters and training data
        for (config_L, config_m, n_train) in data[["config_L", "config_m", "n_train"]].drop_duplicates().values:
            subset = data[
                (data["config_L"] == config_L) & 
                (data["config_m"] == config_m) & 
                (data["n_train"] == n_train)
            ]
            
            if len(subset) < 3:
                continue
            
            # Calculate mean accuracy by context size
            context_acc = subset.groupby("context_size")["accuracy"].agg(["mean", "std", "count"]).reset_index()
            
            if len(context_acc) < 3:
                continue
            
            key = f"L{config_L}_m{config_m}_n{n_train}"
            
            # Fit different curves (emergence patterns)
            x = context_acc["context_size"].values
            y = context_acc["mean"].values
            
            logistic_fit = self.curve_fitter.fit_curve(x, y, "logistic")
            power_fit = self.curve_fitter.fit_curve(x, y, "power")
            
            # Calculate emergence metrics
            emergence_threshold = self._calculate_emergence_threshold_from_data(context_acc)
            
            context_results[key] = {
                "data_points": len(context_acc),
                "context_range": [int(x.min()), int(x.max())],
                "accuracy_range": [float(y.min()), float(y.max())],
                "emergence_threshold": emergence_threshold,
                "logistic_fit": logistic_fit.__dict__,
                "power_fit": power_fit.__dict__,
                "raw_data": context_acc.to_dict("records")
            }
        
        return context_results
    
    def _analyze_combined_scaling(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze combined scaling relationships."""
        combined_results = {}
        
        # Multi-dimensional scaling analysis
        # Create summary data for each configuration
        config_summary = data.groupby(["config_L", "config_m", "n_train", "context_size"]).agg({
            "accuracy": ["mean", "std", "count"],
            "model_params": "first",
            "data_per_param": "first"
        }).reset_index()
        
        # Flatten column names
        config_summary.columns = ["_".join(col).strip("_") for col in config_summary.columns]
        
        # Analyze scaling laws with multiple variables
        scaling_metrics = self._calculate_scaling_metrics(config_summary)
        combined_results["scaling_metrics"] = scaling_metrics
        
        # Analyze optimal ratios
        optimal_ratios = self._analyze_optimal_ratios(config_summary)
        combined_results["optimal_ratios"] = optimal_ratios
        
        return combined_results
    
    def _calculate_scaling_metrics(self, summary_data: pd.DataFrame) -> dict[str, t.Any]:
        """Calculate various scaling metrics."""
        metrics = {}
        
        # Correlation matrix
        numeric_cols = ["config_L", "config_m", "n_train", "context_size", "accuracy_mean", "model_params", "data_per_param"]
        correlation_matrix = summary_data[numeric_cols].corr()
        metrics["correlation_matrix"] = correlation_matrix.to_dict()
        
        # Partial correlations (controlling for other variables)
        try:
            from scipy.stats import pearsonr
            
            # Accuracy vs model params (controlling for training data)
            for n_train in summary_data["n_train"].unique():
                subset = summary_data[summary_data["n_train"] == n_train]
                if len(subset) > 3:
                    corr, p_val = pearsonr(subset["model_params"], subset["accuracy_mean"])
                    metrics[f"params_accuracy_corr_n{n_train}"] = {"correlation": corr, "p_value": p_val}
            
            # Accuracy vs training data (controlling for model size)
            for model_params in summary_data["model_params"].unique():
                subset = summary_data[summary_data["model_params"] == model_params]
                if len(subset) > 3:
                    corr, p_val = pearsonr(subset["n_train"], subset["accuracy_mean"])
                    metrics[f"ntrain_accuracy_corr_p{model_params}"] = {"correlation": corr, "p_value": p_val}
        
        except ImportError:
            pass
        
        return metrics
    
    def _analyze_optimal_ratios(self, summary_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze optimal parameter ratios."""
        ratios = {}
        
        # Find best performing configurations
        best_configs = summary_data.nlargest(10, "accuracy_mean")
        
        if len(best_configs) > 0:
            ratios["best_performing"] = {
                "mean_L": float(best_configs["config_L"].mean()),
                "mean_m": float(best_configs["config_m"].mean()),
                "mean_n_train": float(best_configs["n_train"].mean()),
                "mean_data_per_param": float(best_configs["data_per_param"].mean()),
                "configurations": best_configs[["config_L", "config_m", "n_train", "accuracy_mean"]].to_dict("records")
            }
        
        # Analyze L/m ratio effects
        summary_data["L_m_ratio"] = summary_data["config_L"] / summary_data["config_m"]
        ratio_analysis = summary_data.groupby("L_m_ratio")["accuracy_mean"].agg(["mean", "std", "count"])
        ratios["L_m_ratio_analysis"] = ratio_analysis.to_dict("index")
        
        return ratios
    
    def _fit_scaling_laws(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Fit comprehensive scaling laws."""
        scaling_laws = {}
        
        # Aggregate data for scaling law fitting
        agg_data = data.groupby(["model_params", "n_train", "context_size"]).agg({
            "accuracy": ["mean", "std", "count"]
        }).reset_index()
        
        agg_data.columns = ["model_params", "n_train", "context_size", "accuracy", "accuracy_std", "count"]
        
        # Filter for sufficient samples
        agg_data = agg_data[agg_data["count"] >= self.settings.min_samples_per_condition]
        
        if len(agg_data) < 10:
            return {"error": "Insufficient data for scaling law fitting"}
        
        # Multi-variate scaling law: accuracy = f(params, n_train, context_size)
        try:
            # Log-linear model: log(accuracy) ~ log(params) + log(n_train) + log(context_size)
            from sklearn.linear_model import LinearRegression
            from sklearn.metrics import r2_score
            
            # Prepare features
            X = np.column_stack([
                np.log10(agg_data["model_params"]),
                np.log10(agg_data["n_train"]),
                np.log10(agg_data["context_size"])
            ])
            
            # Use logit transform for accuracy to handle [0,1] range
            y_logit = np.log(agg_data["accuracy"] / (1 - agg_data["accuracy"] + 1e-10))
            
            # Fit model
            model = LinearRegression()
            model.fit(X, y_logit)
            
            y_pred = model.predict(X)
            r2 = r2_score(y_logit, y_pred)
            
            scaling_laws["multivariate_law"] = {
                "success": True,
                "r2": float(r2),
                "coefficients": {
                    "log_params": float(model.coef_[0]),
                    "log_n_train": float(model.coef_[1]),
                    "log_context_size": float(model.coef_[2]),
                    "intercept": float(model.intercept_)
                },
                "equation": f"logit(accuracy) = {model.intercept_:.3f} + {model.coef_[0]:.3f}*log(params) + {model.coef_[1]:.3f}*log(n_train) + {model.coef_[2]:.3f}*log(context)"
            }
            
        except Exception as e:
            scaling_laws["multivariate_law"] = {"success": False, "error": str(e)}
        
        # Simple power laws for each dimension
        scaling_laws["individual_laws"] = self._fit_individual_scaling_laws(agg_data)
        
        return scaling_laws
    
    def _fit_individual_scaling_laws(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Fit individual scaling laws for each dimension."""
        individual_laws = {}
        
        # Parameters vs accuracy (averaged across other dimensions)
        param_avg = data.groupby("model_params")["accuracy"].mean()
        if len(param_avg) >= 3:
            x, y = param_avg.index.values, param_avg.values
            fit = self.curve_fitter.fit_curve(x, y, "power")
            individual_laws["params_law"] = fit.__dict__
        
        # Training data vs accuracy
        ntrain_avg = data.groupby("n_train")["accuracy"].mean()
        if len(ntrain_avg) >= 3:
            x, y = ntrain_avg.index.values, ntrain_avg.values
            fit = self.curve_fitter.fit_curve(x, y, "power")
            individual_laws["ntrain_law"] = fit.__dict__
        
        # Context size vs accuracy
        context_avg = data.groupby("context_size")["accuracy"].mean()
        if len(context_avg) >= 3:
            x, y = context_avg.index.values, context_avg.values
            fit = self.curve_fitter.fit_curve(x, y, "logistic")
            individual_laws["context_law"] = fit.__dict__
        
        return individual_laws
    
    def _analyze_parameter_interactions(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze interactions between different parameters."""
        interactions = {}
        
        # L vs m interaction
        interactions["L_m_interaction"] = self._analyze_L_m_interaction(data)
        
        # Parameter vs data ratio effects
        interactions["data_param_ratio"] = self._analyze_data_param_ratio(data)
        
        # Context size vs model size interaction
        interactions["context_model_interaction"] = self._analyze_context_model_interaction(data)
        
        return interactions
    
    def _analyze_L_m_interaction(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze interaction between L and m parameters."""
        interaction_results = {}
        
        # Create L vs m heatmap data
        lm_accuracy = data.groupby(["config_L", "config_m"])["accuracy"].mean().reset_index()
        lm_pivot = lm_accuracy.pivot(index="config_L", columns="config_m", values="accuracy")
        
        interaction_results["heatmap_data"] = lm_pivot.to_dict()
        
        # Statistical analysis of L vs m effects
        l_effects = data.groupby("config_L")["accuracy"].mean()
        m_effects = data.groupby("config_m")["accuracy"].mean()
        
        interaction_results["main_effects"] = {
            "L_effects": l_effects.to_dict(),
            "m_effects": m_effects.to_dict()
        }
        
        # Test if interaction is significant
        try:
            from scipy.stats import f_oneway
            
            # ANOVA-like analysis for interaction
            groups = []
            for (L, m), group in data.groupby(["config_L", "config_m"]):
                if len(group) >= 3:
                    groups.append(group["accuracy"].values)
            
            if len(groups) >= 2:
                f_stat, p_val = f_oneway(*groups)
                interaction_results["interaction_test"] = {
                    "f_statistic": float(f_stat),
                    "p_value": float(p_val),
                    "significant": p_val < self.settings.significance_threshold
                }
        
        except Exception as e:
            interaction_results["interaction_test"] = {"error": str(e)}
        
        return interaction_results
    
    def _analyze_data_param_ratio(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze effects of data-to-parameter ratio."""
        ratio_results = {}
        
        # Bin data by data-per-parameter ratio
        data["ratio_bin"] = pd.qcut(data["data_per_param"], q=5, labels=["very_low", "low", "medium", "high", "very_high"])
        
        ratio_stats = data.groupby("ratio_bin")["accuracy"].agg(["mean", "std", "count"])
        ratio_results["ratio_stats"] = ratio_stats.to_dict("index")
        
        # Test for trend
        ratio_means = data.groupby("data_per_param")["accuracy"].mean()
        if len(ratio_means) >= 3:
            from scipy.stats import spearmanr
            corr, p_val = spearmanr(ratio_means.index, ratio_means.values)
            ratio_results["trend_test"] = {
                "correlation": float(corr),
                "p_value": float(p_val),
                "significant": p_val < self.settings.significance_threshold
            }
        
        return ratio_results
    
    def _analyze_context_model_interaction(self, data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze interaction between context size and model size."""
        interaction_results = {}
        
        # Group by model size and analyze context scaling
        for model_params in sorted(data["model_params"].unique()):
            subset = data[data["model_params"] == model_params]
            context_scaling = subset.groupby("context_size")["accuracy"].mean()
            
            if len(context_scaling) >= 3:
                # Fit emergence curve
                x, y = context_scaling.index.values, context_scaling.values
                fit = self.curve_fitter.fit_curve(x, y, "logistic")
                
                interaction_results[f"model_params_{model_params}"] = {
                    "context_scaling": context_scaling.to_dict(),
                    "emergence_fit": fit.__dict__
                }
        
        return interaction_results
    
    def _calculate_emergence_threshold_from_data(self, context_acc: pd.DataFrame) -> float:
        """Calculate emergence threshold from context-accuracy data."""
        threshold = 0.5
        
        # Find first context size with accuracy > threshold
        above_threshold = context_acc[context_acc["mean"] > threshold]
        if len(above_threshold) > 0:
            return float(above_threshold["context_size"].min())
        
        # If no emergence, return max + 1
        return float(context_acc["context_size"].max() + 1)
    
    def _create_visualizations(self, data: pd.DataFrame, scaling_results: dict) -> dict[str, str]:
        """Create scaling analysis visualizations."""
        figures = {}
        
        # 1. Multi-panel scaling overview
        fig = self._create_scaling_overview(data)
        figures["scaling_overview"] = str(self.save_figure(fig, "scaling_overview"))
        
        # 2. Parameter scaling heatmaps
        fig = self._create_parameter_heatmaps(data)
        figures["parameter_heatmaps"] = str(self.save_figure(fig, "parameter_heatmaps"))
        
        # 3. Scaling laws visualization
        fig = self._create_scaling_laws_plot(data)
        figures["scaling_laws"] = str(self.save_figure(fig, "scaling_laws"))
        
        # 4. Interaction analysis plots
        fig = self._create_interaction_plots(data)
        figures["interaction_plots"] = str(self.save_figure(fig, "interaction_plots"))
        
        # 5. Optimal ratios visualization
        fig = self._create_optimal_ratios_plot(data)
        figures["optimal_ratios"] = str(self.save_figure(fig, "optimal_ratios"))
        
        return figures
    
    def _create_scaling_overview(self, data: pd.DataFrame) -> plt.Figure:
        """Create comprehensive scaling overview."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Accuracy vs Model Parameters
        sns.scatterplot(data=data, x="model_params", y="accuracy", hue="config_L", 
                       style="config_m", ax=axes[0, 0], alpha=0.7)
        axes[0, 0].set_xscale("log")
        axes[0, 0].set_title("Accuracy vs Model Parameters")
        axes[0, 0].set_xlabel("Model Parameters (L × m)")
        
        # 2. Accuracy vs Training Data
        sns.scatterplot(data=data, x="n_train", y="accuracy", hue="config_L",
                       style="config_m", ax=axes[0, 1], alpha=0.7)
        axes[0, 1].set_xscale("log")
        axes[0, 1].set_title("Accuracy vs Training Data")
        axes[0, 1].set_xlabel("Training Examples")
        
        # 3. Accuracy vs Context Size
        sns.lineplot(data=data, x="context_size", y="accuracy", hue="model_params",
                    ax=axes[0, 2], marker="o")
        axes[0, 2].set_title("Accuracy vs Context Size")
        axes[0, 2].set_xlabel("Context Size")
        
        # 4. Data per Parameter vs Accuracy
        sns.scatterplot(data=data, x="data_per_param", y="accuracy", 
                       hue="context_size", ax=axes[1, 0], alpha=0.7)
        axes[1, 0].set_xscale("log")
        axes[1, 0].set_title("Accuracy vs Data-per-Parameter Ratio")
        axes[1, 0].set_xlabel("Training Data / Parameters")
        
        # 5. L vs m Effects
        lm_pivot = data.groupby(["config_L", "config_m"])["accuracy"].mean().reset_index().pivot(
            index="config_L", columns="config_m", values="accuracy")
        sns.heatmap(lm_pivot, annot=True, fmt=".3f", ax=axes[1, 1], cmap="viridis")
        axes[1, 1].set_title("L × m Configuration Effects")
        
        # 6. Distribution of accuracies
        sns.histplot(data=data, x="accuracy", hue="control_type", ax=axes[1, 2], bins=30)
        axes[1, 2].set_title("Accuracy Distribution")
        
        plt.tight_layout()
        return fig
    
    def _create_parameter_heatmaps(self, data: pd.DataFrame) -> plt.Figure:
        """Create parameter interaction heatmaps."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        # 1. L vs m accuracy heatmap
        lm_pivot = data.groupby(["config_L", "config_m"])["accuracy"].mean().reset_index().pivot(
            index="config_L", columns="config_m", values="accuracy")
        sns.heatmap(lm_pivot, annot=True, fmt=".3f", ax=axes[0, 0], cmap="RdYlBu_r")
        axes[0, 0].set_title("Mean Accuracy by L × m Configuration")
        
        # 2. L vs m sample count heatmap
        lm_count = data.groupby(["config_L", "config_m"]).size().reset_index(name="count").pivot(
            index="config_L", columns="config_m", values="count")
        sns.heatmap(lm_count, annot=True, fmt="d", ax=axes[0, 1], cmap="Blues")
        axes[0, 1].set_title("Sample Count by L × m Configuration")
        
        # 3. Training data vs Model params heatmap
        data["n_train_bin"] = pd.qcut(data["n_train"], q=5, labels=False)
        data["params_bin"] = pd.qcut(data["model_params"], q=5, labels=False)
        np_pivot = data.groupby(["params_bin", "n_train_bin"])["accuracy"].mean().reset_index().pivot(
            index="params_bin", columns="n_train_bin", values="accuracy")
        sns.heatmap(np_pivot, annot=True, fmt=".3f", ax=axes[1, 0], cmap="RdYlBu_r")
        axes[1, 0].set_title("Accuracy by Parameters × Training Data")
        axes[1, 0].set_xlabel("Training Data Quintile")
        axes[1, 0].set_ylabel("Model Parameters Quintile")
        
        # 4. Context vs Model params heatmap
        data["context_bin"] = pd.qcut(data["context_size"], q=min(5, data["context_size"].nunique()), labels=False)
        cp_pivot = data.groupby(["params_bin", "context_bin"])["accuracy"].mean().reset_index().pivot(
            index="params_bin", columns="context_bin", values="accuracy")
        sns.heatmap(cp_pivot, annot=True, fmt=".3f", ax=axes[1, 1], cmap="RdYlBu_r")
        axes[1, 1].set_title("Accuracy by Parameters × Context Size")
        axes[1, 1].set_xlabel("Context Size Quintile")
        axes[1, 1].set_ylabel("Model Parameters Quintile")
        
        plt.tight_layout()
        return fig
    
    def _create_scaling_laws_plot(self, data: pd.DataFrame) -> plt.Figure:
        """Create scaling laws visualization."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Aggregate data for cleaner plots
        param_agg = data.groupby("model_params")["accuracy"].agg(["mean", "std"]).reset_index()
        ntrain_agg = data.groupby("n_train")["accuracy"].agg(["mean", "std"]).reset_index()
        context_agg = data.groupby("context_size")["accuracy"].agg(["mean", "std"]).reset_index()
        
        # 1. Parameter scaling (log-log plot)
        axes[0, 0].errorbar(param_agg["model_params"], param_agg["mean"], 
                           yerr=param_agg["std"], fmt="o-", alpha=0.7)
        axes[0, 0].set_xscale("log")
        axes[0, 0].set_xlabel("Model Parameters")
        axes[0, 0].set_ylabel("Accuracy")
        axes[0, 0].set_title("Parameter Scaling Law")
        axes[0, 0].grid(True, alpha=0.3)
        
        # Fit and plot power law
        if len(param_agg) >= 3:
            x, y = param_agg["model_params"].values, param_agg["mean"].values
            fit = self.curve_fitter.fit_curve(x, y, "power")
            if fit.success:
                x_smooth = np.logspace(np.log10(x.min()), np.log10(x.max()), 100)
                y_smooth = fit.parameters["a"] * np.power(x_smooth, fit.parameters["b"])
                axes[0, 0].plot(x_smooth, y_smooth, "--", alpha=0.8, 
                               label=f"Power law: R²={fit.r2:.3f}")
                axes[0, 0].legend()
        
        # 2. Training data scaling (log-linear plot)
        axes[0, 1].errorbar(ntrain_agg["n_train"], ntrain_agg["mean"],
                           yerr=ntrain_agg["std"], fmt="o-", alpha=0.7)
        axes[0, 1].set_xscale("log")
        axes[0, 1].set_xlabel("Training Examples")
        axes[0, 1].set_ylabel("Accuracy")
        axes[0, 1].set_title("Training Data Scaling Law")
        axes[0, 1].grid(True, alpha=0.3)
        
        # Fit and plot power law
        if len(ntrain_agg) >= 3:
            x, y = ntrain_agg["n_train"].values, ntrain_agg["mean"].values
            fit = self.curve_fitter.fit_curve(x, y, "power")
            if fit.success:
                x_smooth = np.logspace(np.log10(x.min()), np.log10(x.max()), 100)
                y_smooth = fit.parameters["a"] * np.power(x_smooth, fit.parameters["b"])
                axes[0, 1].plot(x_smooth, y_smooth, "--", alpha=0.8,
                               label=f"Power law: R²={fit.r2:.3f}")
                axes[0, 1].legend()
        
        # 3. Context scaling (emergence curve)
        axes[1, 0].errorbar(context_agg["context_size"], context_agg["mean"],
                           yerr=context_agg["std"], fmt="o-", alpha=0.7)
        axes[1, 0].axhline(y=0.5, color="red", linestyle="--", alpha=0.7, label="Emergence threshold")
        axes[1, 0].set_xlabel("Context Size")
        axes[1, 0].set_ylabel("Accuracy")
        axes[1, 0].set_title("Context Size Scaling (Emergence)")
        axes[1, 0].grid(True, alpha=0.3)
        
        # Fit and plot logistic curve
        if len(context_agg) >= 3:
            x, y = context_agg["context_size"].values, context_agg["mean"].values
            fit = self.curve_fitter.fit_curve(x, y, "logistic")
            if fit.success:
                x_smooth = np.linspace(x.min(), x.max(), 100)
                L, k, x0 = fit.parameters["L"], fit.parameters["k"], fit.parameters["x0"]
                y_smooth = L / (1 + np.exp(-k * (x_smooth - x0)))
                axes[1, 0].plot(x_smooth, y_smooth, "--", alpha=0.8,
                               label=f"Logistic: R²={fit.r2:.3f}")
        axes[1, 0].legend()
        
        # 4. Combined scaling (3D-like visualization using data-per-param ratio)
        scatter = axes[1, 1].scatter(data["data_per_param"], data["accuracy"], 
                                   c=data["context_size"], s=data["model_params"]/10,
                                   alpha=0.6, cmap="viridis")
        axes[1, 1].set_xscale("log")
        axes[1, 1].set_xlabel("Data per Parameter")
        axes[1, 1].set_ylabel("Accuracy")
        axes[1, 1].set_title("Combined Scaling\n(Size=Model Params, Color=Context Size)")
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=axes[1, 1])
        cbar.set_label("Context Size")
        
        plt.tight_layout()
        return fig
    
    def _create_interaction_plots(self, data: pd.DataFrame) -> plt.Figure:
        """Create parameter interaction visualization."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. L vs m interaction (grouped by context size)
        for context_size in sorted(data["context_size"].unique())[:5]:  # Limit to first 5 for clarity
            subset = data[data["context_size"] == context_size]
            lm_acc = subset.groupby(["config_L", "config_m"])["accuracy"].mean().reset_index()
            
            # Create interaction plot
            for m in sorted(lm_acc["config_m"].unique()):
                m_data = lm_acc[lm_acc["config_m"] == m]
                axes[0, 0].plot(m_data["config_L"], m_data["accuracy"], 
                               marker="o", label=f"k={context_size}, m={m}", alpha=0.7)
        
        axes[0, 0].set_xlabel("Sequence Length (L)")
        axes[0, 0].set_ylabel("Accuracy")
        axes[0, 0].set_title("L × m Interaction by Context Size")
        axes[0, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Data-per-parameter effects by model size
        data["model_size_bin"] = pd.qcut(data["model_params"], q=4, labels=["Small", "Medium", "Large", "XLarge"])
        
        for size_bin in data["model_size_bin"].unique():
            if pd.notna(size_bin):
                subset = data[data["model_size_bin"] == size_bin]
                ratio_acc = subset.groupby("data_per_param")["accuracy"].mean()
                axes[0, 1].plot(ratio_acc.index, ratio_acc.values, 
                               marker="o", label=f"Model: {size_bin}", alpha=0.7)
        
        axes[0, 1].set_xscale("log")
        axes[0, 1].set_xlabel("Data per Parameter")
        axes[0, 1].set_ylabel("Accuracy")
        axes[0, 1].set_title("Data Efficiency by Model Size")
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Context scaling by model parameters
        param_values = sorted(data["model_params"].unique())[:6]  # Limit for clarity
        
        for params in param_values:
            subset = data[data["model_params"] == params]
            context_acc = subset.groupby("context_size")["accuracy"].mean()
            axes[1, 0].plot(context_acc.index, context_acc.values,
                           marker="o", label=f"Params={params}", alpha=0.7)
        
        axes[1, 0].axhline(y=0.5, color="red", linestyle="--", alpha=0.5)
        axes[1, 0].set_xlabel("Context Size")
        axes[1, 0].set_ylabel("Accuracy")
        axes[1, 0].set_title("Emergence by Model Size")
        axes[1, 0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Training data scaling by context size
        context_values = sorted(data["context_size"].unique())[:5]
        
        for context in context_values:
            subset = data[data["context_size"] == context]
            ntrain_acc = subset.groupby("n_train")["accuracy"].mean()
            axes[1, 1].plot(ntrain_acc.index, ntrain_acc.values,
                           marker="o", label=f"Context={context}", alpha=0.7)
        
        axes[1, 1].set_xscale("log")
        axes[1, 1].set_xlabel("Training Examples")
        axes[1, 1].set_ylabel("Accuracy")
        axes[1, 1].set_title("Training Scaling by Context Size")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    def _create_optimal_ratios_plot(self, data: pd.DataFrame) -> plt.Figure:
        """Create optimal ratios visualization."""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. L/m ratio effects
        data["L_m_ratio"] = data["config_L"] / data["config_m"]
        ratio_acc = data.groupby("L_m_ratio")["accuracy"].agg(["mean", "std", "count"]).reset_index()
        
        axes[0, 0].errorbar(ratio_acc["L_m_ratio"], ratio_acc["mean"],
                           yerr=ratio_acc["std"], fmt="o-", alpha=0.7)
        axes[0, 0].set_xlabel("L/m Ratio")
        axes[0, 0].set_ylabel("Mean Accuracy")
        axes[0, 0].set_title("Effect of L/m Ratio")
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Data-per-parameter optimal range
        data["ratio_bin"] = pd.qcut(data["data_per_param"], q=10, duplicates='drop')
        ratio_stats = data.groupby("ratio_bin")["accuracy"].agg(["mean", "std", "count"]).reset_index()
        
        # Get bin centers for plotting
        bin_centers = ratio_stats["ratio_bin"].apply(lambda x: x.mid)
        axes[0, 1].errorbar(bin_centers, ratio_stats["mean"], 
                           yerr=ratio_stats["std"], fmt="o-", alpha=0.7)
        axes[0, 1].set_xscale("log")
        axes[0, 1].set_xlabel("Data per Parameter")
        axes[0, 1].set_ylabel("Mean Accuracy")
        axes[0, 1].set_title("Optimal Data-per-Parameter Range")
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Best performing configurations
        config_performance = data.groupby(["config_L", "config_m", "n_train"])["accuracy"].mean().reset_index()
        top_configs = config_performance.nlargest(10, "accuracy")
        
        scatter = axes[1, 0].scatter(top_configs["config_L"], top_configs["config_m"],
                                    s=top_configs["n_train"]/10, c=top_configs["accuracy"],
                                    cmap="viridis", alpha=0.7)
        axes[1, 0].set_xlabel("Sequence Length (L)")
        axes[1, 0].set_ylabel("Number of Features (m)")
        axes[1, 0].set_title("Top 10 Configurations\n(Size=Training Data, Color=Accuracy)")
        
        cbar = plt.colorbar(scatter, ax=axes[1, 0])
        cbar.set_label("Accuracy")
        
        # 4. Parameter efficiency frontier
        # Plot Pareto frontier of accuracy vs model parameters
        config_summary = data.groupby(["config_L", "config_m"]).agg({
            "accuracy": "mean",
            "model_params": "first"
        }).reset_index()
        
        # Find Pareto efficient points
        pareto_points = self._find_pareto_frontier(config_summary[["model_params", "accuracy"]].values)
        pareto_data = config_summary.iloc[pareto_points]
        
        axes[1, 1].scatter(config_summary["model_params"], config_summary["accuracy"],
                          alpha=0.5, label="All configs")
        axes[1, 1].scatter(pareto_data["model_params"], pareto_data["accuracy"],
                          color="red", s=60, label="Pareto frontier", zorder=5)
        
        # Connect Pareto points
        pareto_sorted = pareto_data.sort_values("model_params")
        axes[1, 1].plot(pareto_sorted["model_params"], pareto_sorted["accuracy"],
                       "--", color="red", alpha=0.7)
        
        axes[1, 1].set_xlabel("Model Parameters")
        axes[1, 1].set_ylabel("Accuracy")
        axes[1, 1].set_title("Parameter Efficiency Frontier")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig
    
    def _find_pareto_frontier(self, points: np.ndarray) -> list[int]:
        """Find Pareto efficient points (minimize parameters, maximize accuracy)."""
        # Convert to minimize both objectives (negate accuracy)
        objectives = points.copy()
        objectives[:, 1] = -objectives[:, 1]  # Negate accuracy for minimization
        
        pareto_indices = []
        for i, point in enumerate(objectives):
            is_pareto = True
            for j, other_point in enumerate(objectives):
                if i != j:
                    # Check if other point dominates this point
                    if all(other_point <= point) and any(other_point < point):
                        is_pareto = False
                        break
            if is_pareto:
                pareto_indices.append(i)
        
        return pareto_indices
    
    def _create_summary(self, scaling_results: dict, scaling_laws: dict) -> dict[str, t.Any]:
        """Create scaling analysis summary."""
        summary = {}
        
        # Count successful analyses
        param_analyses = len([r for r in scaling_results.get("parameter_scaling", {}).values() 
                             if isinstance(r, dict) and "power_law_fit" in r])
        data_analyses = len([r for r in scaling_results.get("training_data_scaling", {}).values()
                            if isinstance(r, dict) and "power_law_fit" in r])
        context_analyses = len([r for r in scaling_results.get("context_scaling", {}).values()
                               if isinstance(r, dict) and "logistic_fit" in r])
        
        summary["analysis_counts"] = {
            "parameter_scaling_analyses": param_analyses,
            "training_data_analyses": data_analyses,
            "context_scaling_analyses": context_analyses
        }
        
        # Extract key scaling law coefficients
        if "multivariate_law" in scaling_laws and scaling_laws["multivariate_law"].get("success"):
            mv_law = scaling_laws["multivariate_law"]
            summary["multivariate_scaling"] = {
                "r2": mv_law["r2"],
                "parameter_coefficient": mv_law["coefficients"]["log_params"],
                "data_coefficient": mv_law["coefficients"]["log_n_train"],
                "context_coefficient": mv_law["coefficients"]["log_context_size"]
            }
        
        # Individual law summaries
        individual_laws = scaling_laws.get("individual_laws", {})
        for law_name, law_result in individual_laws.items():
            if law_result.get("success"):
                summary[f"{law_name}_summary"] = {
                    "r2": law_result["r2"],
                    "curve_type": law_result["curve_type"]
                }
        
        # Interaction summaries
        interactions = scaling_results.get("parameter_interactions", {})
        if "L_m_interaction" in interactions:
            lm_interaction = interactions["L_m_interaction"]
            if "interaction_test" in lm_interaction:
                summary["L_m_interaction_significant"] = lm_interaction["interaction_test"].get("significant", False)
        
        return summary
    
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate comprehensive scaling laws report."""
        report_path = self.output_dir / "reports" / "rq2_scaling_report.md"
        
        with open(report_path, "w") as f:
            f.write("# RQ2: ICL Scaling Laws Analysis Report\n\n")
            f.write(f"Generated: {pd.Timestamp.now()}\n\n")
            
            # Executive Summary
            f.write("## Executive Summary\n\n")
            summary = results["summary"]
            
            f.write(f"- **Parameter Scaling Analyses**: {summary['analysis_counts']['parameter_scaling_analyses']}\n")
            f.write(f"- **Training Data Analyses**: {summary['analysis_counts']['training_data_analyses']}\n")
            f.write(f"- **Context Scaling Analyses**: {summary['analysis_counts']['context_scaling_analyses']}\n\n")
            
            # Multivariate scaling law
            if "multivariate_scaling" in summary:
                mv = summary["multivariate_scaling"]
                f.write("### Multivariate Scaling Law\n")
                f.write(f"- **Overall Fit (R²)**: {mv['r2']:.4f}\n")
                f.write(f"- **Parameter Scaling Coefficient**: {mv['parameter_coefficient']:.4f}\n")
                f.write(f"- **Training Data Coefficient**: {mv['data_coefficient']:.4f}\n")
                f.write(f"- **Context Size Coefficient**: {mv['context_coefficient']:.4f}\n\n")
            
            # Key findings
            f.write("## Key Findings\n\n")
            
            # Parameter scaling findings
            param_scaling = results["scaling_analysis"].get("parameter_scaling", {})
            if param_scaling:
                f.write("### Parameter Scaling\n")
                successful_fits = sum(1 for r in param_scaling.values() 
                                    if isinstance(r, dict) and r.get("power_law_fit", {}).get("success"))
                f.write(f"- Successful power law fits: {successful_fits}/{len(param_scaling)}\n")
                
                # Average scaling exponent
                exponents = []
                for result in param_scaling.values():
                    if isinstance(result, dict) and result.get("power_law_fit", {}).get("success"):
                        params = result["power_law_fit"].get("parameters", {})
                        if "b" in params:
                            exponents.append(params["b"])
                
                if exponents:
                    f.write(f"- Average scaling exponent: {np.mean(exponents):.4f} ± {np.std(exponents):.4f}\n")
                f.write("\n")
            
            # Training data scaling findings
            data_scaling = results["scaling_analysis"].get("training_data_scaling", {})
            if data_scaling:
                f.write("### Training Data Scaling\n")
                successful_fits = sum(1 for r in data_scaling.values()
                                    if isinstance(r, dict) and r.get("power_law_fit", {}).get("success"))
                f.write(f"- Successful power law fits: {successful_fits}/{len(data_scaling)}\n")
                
                # Average scaling exponent
                exponents = []
                for result in data_scaling.values():
                    if isinstance(result, dict) and result.get("power_law_fit", {}).get("success"):
                        params = result["power_law_fit"].get("parameters", {})
                        if "b" in params:
                            exponents.append(params["b"])
                
                if exponents:
                    f.write(f"- Average scaling exponent: {np.mean(exponents):.4f} ± {np.std(exponents):.4f}\n")
                f.write("\n")
            
            # Context scaling findings
            context_scaling = results["scaling_analysis"].get("context_scaling", {})
            if context_scaling:
                f.write("### Context Size Scaling (Emergence)\n")
                successful_fits = sum(1 for r in context_scaling.values()
                                    if isinstance(r, dict) and r.get("logistic_fit", {}).get("success"))
                f.write(f"- Successful logistic fits: {successful_fits}/{len(context_scaling)}\n")
                
                # Average emergence thresholds
                thresholds = []
                for result in context_scaling.values():
                    if isinstance(result, dict) and "emergence_threshold" in result:
                        threshold = result["emergence_threshold"]
                        if threshold != float("inf"):
                            thresholds.append(threshold)
                
                if thresholds:
                    f.write(f"- Average emergence threshold: {np.mean(thresholds):.2f} ± {np.std(thresholds):.2f}\n")
                f.write("\n")
            
            # Parameter interactions
            interactions = results.get("parameter_interactions", {})
            if interactions:
                f.write("### Parameter Interactions\n")
                
                # L vs m interaction
                if "L_m_interaction" in interactions:
                    lm_test = interactions["L_m_interaction"].get("interaction_test", {})
                    if "significant" in lm_test:
                        significance = "significant" if lm_test["significant"] else "not significant"
                        f.write(f"- L × m interaction: {significance} (p={lm_test.get('p_value', 'N/A'):.4f})\n")
                
                # Data-parameter ratio
                if "data_param_ratio" in interactions:
                    ratio_test = interactions["data_param_ratio"].get("trend_test", {})
                    if "significant" in ratio_test:
                        significance = "significant" if ratio_test["significant"] else "not significant"
                        corr = ratio_test.get("correlation", 0)
                        f.write(f"- Data-per-parameter trend: {significance} (ρ={corr:.4f})\n")
                f.write("\n")
            
            # Scaling laws details
            f.write("## Detailed Scaling Laws\n\n")
            scaling_laws = results.get("scaling_laws", {})
            
            if "individual_laws" in scaling_laws:
                for law_name, law_result in scaling_laws["individual_laws"].items():
                    if law_result.get("success"):
                        f.write(f"### {law_name.replace('_', ' ').title()}\n")
                        f.write(f"- **Equation**: {law_result.get('equation', 'N/A')}\n")
                        f.write(f"- **R²**: {law_result.get('r2', 'N/A'):.4f}\n")
                        f.write(f"- **Curve Type**: {law_result.get('curve_type', 'N/A')}\n\n")
            
            # Figures
            f.write("## Generated Figures\n\n")
            for fig_name, fig_path in results["figures"].items():
                f.write(f"- **{fig_name.replace('_', ' ').title()}**: `{fig_path}`\n")
        
        return report_path


# Example usage function
def run_scaling_analysis(
    results_dir: str | Path,
    **analysis_kwargs
) -> dict[str, t.Any]:
    """Run scaling laws analysis with custom settings."""
    results_path = Path(results_dir)
    
    # Setup analysis settings
    settings = AnalysisSettings(**analysis_kwargs)
    
    # Initialize and run analyzer
    analyzer = RQ2ScalingAnalyzer(results_path, settings)
    results = analyzer.run_analysis()
    
    # Save results and generate report
    analyzer.save_results(results)
    report_path = analyzer.generate_report(results)
    
    print(f"Scaling laws analysis completed!")
    print(f"Results saved to: {analyzer.output_dir}")
    print(f"Report: {report_path}")
    
    return results

# RQ3: Attention Pattern

In [ ]:
"""RQ3: Analyze attention patterns and internal representations for ICL mechanistic understanding."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter


@dataclass
class MechanisticConfig(AnalysisConfig):
    """Configuration for mechanistic analysis."""
    # Attention analysis parameters
    attention_layers: list[int] | None = None  # None for all layers
    attention_heads: list[int] | None = None   # None for all heads
    context_positions: list[int] | None = None # None for all positions
    
    # Representation analysis parameters
    probe_hidden_dims: list[int] = field(default_factory=lambda: [64, 128])
    probe_regularization: float = 0.01
    probe_max_iter: int = 1000
    
    # Specialization analysis parameters
    specialization_metric: str = "entropy"  # "entropy", "variance", "gini"
    min_attention_threshold: float = 0.01
    
    # PCA parameters
    pca_components: int = 10
    pca_variance_threshold: float = 0.95
    
    # Clustering parameters
    n_clusters: int = 5
    clustering_method: str = "kmeans"  # "kmeans", "hierarchical"


class MechanisticAnalyzer(BaseAnalyzer):
    """Analyzer for mechanistic understanding of ICL (RQ3)."""
    
    def __init__(self, config: MechanisticConfig):
        """Initialize mechanistic analyzer."""
        super().__init__(config)
        self.config: MechanisticConfig = config
        
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive mechanistic analysis."""
        print("Starting RQ3: Mechanistic Analysis")
        print("=" * 50)
        
        # Load and validate data
        attention_data = self._load_attention_data()
        representation_data = self._load_representation_data()
        performance_data = self._load_performance_data()
        
        print(f"Loaded attention data for {len(attention_data)} evaluations")
        print(f"Loaded representation data for {len(representation_data)} evaluations")
        print(f"Loaded performance data for {len(performance_data)} evaluations")
        
        results = {}
        
        # 1. Attention pattern analysis
        print("\n1. Analyzing attention patterns...")
        attention_analysis = self._analyze_attention_patterns(attention_data, performance_data)
        results["attention_analysis"] = attention_analysis
        
        # 2. Layer specialization analysis
        print("2. Analyzing layer specialization...")
        specialization_analysis = self._analyze_layer_specialization(attention_data)
        results["specialization_analysis"] = specialization_analysis
        
        # 3. Representation analysis
        print("3. Analyzing internal representations...")
        representation_analysis = self._analyze_representations(representation_data, performance_data)
        results["representation_analysis"] = representation_analysis
        
        # 4. Probe training for interpretability
        print("4. Training interpretability probes...")
        probe_analysis = self._train_interpretability_probes(representation_data, performance_data)
        results["probe_analysis"] = probe_analysis
        
        # 5. Mechanistic pattern correlation with performance
        print("5. Correlating patterns with performance...")
        correlation_analysis = self._correlate_patterns_with_performance(results, performance_data)
        results["correlation_analysis"] = correlation_analysis
        
        # 6. Generate visualizations
        print("6. Generating visualizations...")
        self._generate_mechanistic_visualizations(results)
        
        # Save results
        self.save_results(results, "rq3_mechanistic_results.json")
        
        print("\nRQ3 Analysis completed successfully!")
        return results
    
    def _load_attention_data(self) -> pd.DataFrame:
        """Load attention pattern data."""
        # Load attention data from the attention_data directory
        attention_dir = self.config.results_dir / "raw_evaluations" / "attention_data"
        
        if not attention_dir.exists():
            print("Warning: No attention data directory found")
            return pd.DataFrame()
        
        # Load attention metadata
        metadata_path = attention_dir / "metadata.json"
        if metadata_path.exists():
            import json
            with open(metadata_path) as f:
                attention_metadata = json.load(f)
        else:
            attention_metadata = {}
        
        # Load attention patterns for analysis
        attention_records = []
        
        model_dirs = [d for d in attention_dir.iterdir() if d.is_dir() and d.name != "by_model"]
        if (attention_dir / "by_model").exists():
            model_dirs = list((attention_dir / "by_model").iterdir())
        
        for model_dir in model_dirs[:10]:  # Limit to first 10 models for memory
            model_id = model_dir.name
            
            for attention_file in model_dir.glob("*.npz"):
                try:
                    # Load attention weights
                    attention_data = np.load(attention_file)
                    
                    # Extract metadata from filename (assumes format: modelid_configL_configm_ntrain_k_seqid.npz)
                    filename_parts = attention_file.stem.split("_")
                    if len(filename_parts) >= 6:
                        config_L = int(filename_parts[1])
                        config_m = int(filename_parts[2])
                        n_train = int(filename_parts[3])
                        context_size = int(filename_parts[4])
                        sequence_id = int(filename_parts[5])
                        
                        # Extract attention weights (shape: [layers, heads, seq_len, seq_len])
                        attention_weights = attention_data.get("attention_weights")
                        if attention_weights is not None:
                            attention_records.append({
                                "model_id": model_id,
                                "config_L": config_L,
                                "config_m": config_m,
                                "n_train": n_train,
                                "context_size": context_size,
                                "sequence_id": sequence_id,
                                "attention_weights": attention_weights,
                                "n_layers": attention_weights.shape[0],
                                "n_heads": attention_weights.shape[1],
                                "seq_length": attention_weights.shape[2]
                            })
                
                except Exception as e:
                    print(f"Warning: Could not load attention file {attention_file}: {e}")
                    continue
        
        return pd.DataFrame(attention_records)
    
    def _load_representation_data(self) -> pd.DataFrame:
        """Load internal representation data."""
        # Load representation data from the representations directory
        repr_dir = self.config.results_dir / "raw_evaluations" / "representations"
        
        if not repr_dir.exists():
            print("Warning: No representation data directory found")
            return pd.DataFrame()
        
        # Load representation metadata
        metadata_path = repr_dir / "metadata.json"
        if metadata_path.exists():
            import json
            with open(metadata_path) as f:
                repr_metadata = json.load(f)
        else:
            repr_metadata = {}
        
        # Load representation data for analysis
        repr_records = []
        
        layer_dirs = [d for d in repr_dir.iterdir() if d.is_dir() and d.name.startswith("layer_")]
        
        for layer_dir in layer_dirs:
            layer_num = int(layer_dir.name.split("_")[1])
            
            for repr_file in layer_dir.glob("*.npz")[:50]:  # Limit files for memory
                try:
                    # Load hidden states
                    repr_data = np.load(repr_file)
                    
                    # Extract metadata from filename
                    filename_parts = repr_file.stem.split("_")
                    if len(filename_parts) >= 6:
                        model_id = filename_parts[0]
                        config_L = int(filename_parts[1])
                        config_m = int(filename_parts[2])
                        n_train = int(filename_parts[3])
                        context_size = int(filename_parts[4])
                        sequence_id = int(filename_parts[5])
                        
                        # Extract hidden states (shape: [seq_len, hidden_dim])
                        hidden_states = repr_data.get("hidden_states")
                        if hidden_states is not None:
                            repr_records.append({
                                "model_id": model_id,
                                "config_L": config_L,
                                "config_m": config_m,
                                "n_train": n_train,
                                "context_size": context_size,
                                "sequence_id": sequence_id,
                                "layer": layer_num,
                                "hidden_states": hidden_states,
                                "seq_length": hidden_states.shape[0],
                                "hidden_dim": hidden_states.shape[1]
                            })
                
                except Exception as e:
                    print(f"Warning: Could not load representation file {repr_file}: {e}")
                    continue
        
        return pd.DataFrame(repr_records)
    
    def _load_performance_data(self) -> pd.DataFrame:
        """Load performance data for correlation analysis."""
        # Load ICL performance data
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        return data
    
    def _analyze_attention_patterns(self, attention_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze attention patterns and their relationship to ICL performance."""
        if attention_data.empty:
            return {"error": "No attention data available"}
        
        pattern_results = {}
        
        # Analyze attention patterns by configuration
        for (config_L, config_m, n_train), group in attention_data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            
            attention_metrics = []
            
            for _, row in group.iterrows():
                attention_weights = row["attention_weights"]
                
                # Compute various attention metrics
                metrics = self._compute_attention_metrics(attention_weights)
                metrics.update({
                    "model_id": row["model_id"],
                    "context_size": row["context_size"],
                    "sequence_id": row["sequence_id"]
                })
                attention_metrics.append(metrics)
            
            if attention_metrics:
                pattern_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "attention_metrics": attention_metrics,
                    "n_samples": len(attention_metrics)
                }
        
        # Aggregate attention patterns
        aggregated_patterns = self._aggregate_attention_patterns(pattern_results)
        pattern_results["aggregated_patterns"] = aggregated_patterns
        
        return pattern_results
    
    def _compute_attention_metrics(self, attention_weights: np.ndarray) -> dict[str, float]:
        """Compute various metrics from attention weights."""
        n_layers, n_heads, seq_len, _ = attention_weights.shape
        
        metrics = {}
        
        # 1. Attention entropy (measure of attention spread)
        entropies = []
        for layer in range(n_layers):
            for head in range(n_heads):
                attn = attention_weights[layer, head]
                for pos in range(seq_len):
                    attn_dist = attn[pos] + 1e-8  # Avoid log(0)
                    entropy = -np.sum(attn_dist * np.log(attn_dist))
                    entropies.append(entropy)
        
        metrics["mean_attention_entropy"] = float(np.mean(entropies))
        metrics["std_attention_entropy"] = float(np.std(entropies))
        
        # 2. Attention concentration (inverse of entropy)
        metrics["mean_attention_concentration"] = 1.0 / (1.0 + metrics["mean_attention_entropy"])
        
        # 3. Layer-wise attention variance
        layer_variances = []
        for layer in range(n_layers):
            layer_attn = attention_weights[layer].mean(axis=0)  # Average over heads
            layer_variances.append(float(np.var(layer_attn)))
        
        metrics["attention_layer_variance"] = layer_variances
        metrics["mean_layer_variance"] = float(np.mean(layer_variances))
        
        # 4. Head specialization (variance across heads within layers)
        head_specializations = []
        for layer in range(n_layers):
            head_patterns = []
            for head in range(n_heads):
                head_pattern = attention_weights[layer, head].flatten()
                head_patterns.append(head_pattern)
            
            if len(head_patterns) > 1:
                head_matrix = np.stack(head_patterns)
                specialization = float(np.mean(np.var(head_matrix, axis=0)))
                head_specializations.append(specialization)
        
        metrics["head_specialization"] = head_specializations
        metrics["mean_head_specialization"] = float(np.mean(head_specializations)) if head_specializations else 0.0
        
        # 5. Position-wise attention patterns
        position_attentions = []
        for pos in range(seq_len):
            pos_attn = attention_weights[:, :, pos, :].mean(axis=(0, 1))  # Average over layers and heads
            position_attentions.append(pos_attn.tolist())
        
        metrics["position_attention_patterns"] = position_attentions
        
        # 6. Attention to previous tokens vs. future tokens
        if seq_len > 1:
            prev_attention = []
            for layer in range(n_layers):
                for head in range(n_heads):
                    for pos in range(1, seq_len):
                        prev_attn = np.sum(attention_weights[layer, head, pos, :pos])
                        prev_attention.append(prev_attn)
            
            metrics["mean_previous_attention"] = float(np.mean(prev_attention))
            metrics["std_previous_attention"] = float(np.std(prev_attention))
        
        return metrics
    
    def _aggregate_attention_patterns(self, pattern_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Aggregate attention patterns across configurations."""
        aggregated = {
            "entropy_patterns": {},
            "specialization_patterns": {},
            "position_patterns": {}
        }
        
        for group_key, result in pattern_results.items():
            if "attention_metrics" not in result:
                continue
                
            metrics_list = result["attention_metrics"]
            
            # Aggregate entropy metrics
            entropies = [m["mean_attention_entropy"] for m in metrics_list]
            concentrations = [m["mean_attention_concentration"] for m in metrics_list]
            
            aggregated["entropy_patterns"][group_key] = {
                "mean_entropy": float(np.mean(entropies)),
                "std_entropy": float(np.std(entropies)),
                "mean_concentration": float(np.mean(concentrations)),
                "std_concentration": float(np.std(concentrations))
            }
            
            # Aggregate specialization metrics
            specializations = [m["mean_head_specialization"] for m in metrics_list]
            layer_variances = [m["mean_layer_variance"] for m in metrics_list]
            
            aggregated["specialization_patterns"][group_key] = {
                "mean_head_specialization": float(np.mean(specializations)),
                "std_head_specialization": float(np.std(specializations)),
                "mean_layer_variance": float(np.mean(layer_variances)),
                "std_layer_variance": float(np.std(layer_variances))
            }
        
        return aggregated
    
    def _analyze_layer_specialization(self, attention_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how different layers specialize for ICL tasks."""
        if attention_data.empty:
            return {"error": "No attention data available"}
        
        specialization_results = {}
        
        # Analyze layer specialization patterns
        for (config_L, config_m, n_train), group in attention_data.groupby(["config_L", "config_m", "n_train"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}"
            
            layer_metrics = []
            
            for _, row in group.iterrows():
                attention_weights = row["attention_weights"]
                n_layers = attention_weights.shape[0]
                
                # Compute specialization for each layer
                for layer in range(n_layers):
                    layer_attn = attention_weights[layer]
                    
                    # Compute layer specialization metrics
                    specialization = self._compute_layer_specialization(layer_attn, layer)
                    specialization.update({
                        "model_id": row["model_id"],
                        "context_size": row["context_size"],
                        "sequence_id": row["sequence_id"],
                        "layer": layer
                    })
                    layer_metrics.append(specialization)
            
            if layer_metrics:
                # Aggregate by layer
                layer_aggregates = {}
                for layer in range(max([m["layer"] for m in layer_metrics]) + 1):
                    layer_data = [m for m in layer_metrics if m["layer"] == layer]
                    if layer_data:
                        layer_aggregates[layer] = self._aggregate_layer_metrics(layer_data)
                
                specialization_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "layer_specialization": layer_aggregates,
                    "n_layers": len(layer_aggregates)
                }
        
        return specialization_results
    
    def _compute_layer_specialization(self, layer_attention: np.ndarray, layer_idx: int) -> dict[str, float]:
        """Compute specialization metrics for a specific layer."""
        n_heads, seq_len, _ = layer_attention.shape
        
        metrics = {}
        
        # 1. Attention entropy for this layer
        entropies = []
        for head in range(n_heads):
            for pos in range(seq_len):
                attn_dist = layer_attention[head, pos] + 1e-8
                entropy = -np.sum(attn_dist * np.log(attn_dist))
                entropies.append(entropy)
        
        metrics["layer_entropy_mean"] = float(np.mean(entropies))
        metrics["layer_entropy_std"] = float(np.std(entropies))
        
        # 2. Head diversity within layer
        head_patterns = []
        for head in range(n_heads):
            head_pattern = layer_attention[head].flatten()
            head_patterns.append(head_pattern)
        
        if len(head_patterns) > 1:
            head_matrix = np.stack(head_patterns)
            # Compute pairwise correlations between heads
            correlations = np.corrcoef(head_matrix)
            # Mean correlation as inverse of diversity
            mean_correlation = float(np.mean(correlations[np.triu_indices_from(correlations, k=1)]))
            metrics["head_diversity"] = 1.0 - abs(mean_correlation)
        else:
            metrics["head_diversity"] = 0.0
        
        # 3. Position specialization
        position_specializations = []
        for pos in range(seq_len):
            pos_attn = layer_attention[:, pos, :].mean(axis=0)  # Average over heads
            specialization = float(np.max(pos_attn) - np.min(pos_attn))
            position_specializations.append(specialization)
        
        metrics["position_specialization_mean"] = float(np.mean(position_specializations))
        metrics["position_specialization_max"] = float(np.max(position_specializations))
        
        # 4. Layer depth relative specialization
        metrics["layer_depth_ratio"] = float(layer_idx / max(1, n_heads))  # Normalize by number of heads
        
        return metrics
    
    def _aggregate_layer_metrics(self, layer_data: list[dict[str, t.Any]]) -> dict[str, t.Any]:
        """Aggregate metrics for a specific layer across samples."""
        if not layer_data:
            return {}
        
        aggregated = {}
        
        # Get all numeric metrics
        numeric_keys = [k for k in layer_data[0].keys() 
                       if isinstance(layer_data[0][k], (int, float, np.number))]
        
        for key in numeric_keys:
            values = [d[key] for d in layer_data if key in d and d[key] is not None]
            if values:
                aggregated[f"{key}_mean"] = float(np.mean(values))
                aggregated[f"{key}_std"] = float(np.std(values))
                aggregated[f"{key}_min"] = float(np.min(values))
                aggregated[f"{key}_max"] = float(np.max(values))
        
        aggregated["n_samples"] = len(layer_data)
        return aggregated
    
    def _analyze_representations(self, representation_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze internal representations and their evolution."""
        if representation_data.empty:
            return {"error": "No representation data available"}
        
        repr_results = {}
        
        # Analyze representations by configuration and layer
        for (config_L, config_m, n_train, layer), group in representation_data.groupby(["config_L", "config_m", "n_train", "layer"]):
            group_key = f"L{config_L}_m{config_m}_n{n_train}_layer{layer}"
            
            repr_metrics = []
            
            for _, row in group.iterrows():
                hidden_states = row["hidden_states"]
                
                # Compute representation metrics
                metrics = self._compute_representation_metrics(hidden_states, layer)
                metrics.update({
                    "model_id": row["model_id"],
                    "context_size": row["context_size"],
                    "sequence_id": row["sequence_id"]
                })
                repr_metrics.append(metrics)
            
            if repr_metrics:
                # Aggregate metrics for this configuration/layer
                aggregated = self._aggregate_representation_metrics(repr_metrics)
                
                repr_results[group_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "n_train": n_train,
                    "layer": layer,
                    "representation_metrics": aggregated,
                    "n_samples": len(repr_metrics)
                }
        
        # Analyze representation evolution across layers
        layer_evolution = self._analyze_layer_evolution(repr_results)
        repr_results["layer_evolution"] = layer_evolution
        
        return repr_results
    
    def _compute_representation_metrics(self, hidden_states: np.ndarray, layer: int) -> dict[str, float]:
        """Compute metrics from hidden state representations."""
        seq_len, hidden_dim = hidden_states.shape
        
        metrics = {}
        
        # 1. Representation norm and variance
        norms = np.linalg.norm(hidden_states, axis=1)
        metrics["mean_norm"] = float(np.mean(norms))
        metrics["std_norm"] = float(np.std(norms))
        
        # 2. Dimensionality and effective rank
        # Use SVD to estimate effective dimensionality
        try:
            U, s, Vt = np.linalg.svd(hidden_states, full_matrices=False)
            # Effective rank based on singular value distribution
            s_normalized = s / np.sum(s)
            entropy = -np.sum(s_normalized * np.log(s_normalized + 1e-8))
            metrics["effective_dimensionality"] = float(np.exp(entropy))
            metrics["rank_ratio"] = float(metrics["effective_dimensionality"] / min(seq_len, hidden_dim))
        except:
            metrics["effective_dimensionality"] = float(min(seq_len, hidden_dim))
            metrics["rank_ratio"] = 1.0
        
        # 3. Position-wise representation similarity
        if seq_len > 1:
            similarities = []
            for i in range(seq_len - 1):
                sim = np.dot(hidden_states[i], hidden_states[i + 1]) / (
                    np.linalg.norm(hidden_states[i]) * np.linalg.norm(hidden_states[i + 1]) + 1e-8
                )
                similarities.append(sim)
            
            metrics["mean_position_similarity"] = float(np.mean(similarities))
            metrics["std_position_similarity"] = float(np.std(similarities))
        
        # 4. Representation clustering tendency
        if seq_len > 2:
            # Compute pairwise distances
            distances = []
            for i in range(seq_len):
                for j in range(i + 1, seq_len):
                    dist = np.linalg.norm(hidden_states[i] - hidden_states[j])
                    distances.append(dist)
            
            metrics["mean_pairwise_distance"] = float(np.mean(distances))
            metrics["std_pairwise_distance"] = float(np.std(distances))
        
        # 5. Layer depth indicator
        metrics["layer_depth"] = float(layer)
        
        return metrics
    
    def _aggregate_representation_metrics(self, repr_metrics: list[dict[str, t.Any]]) -> dict[str, t.Any]:
        """Aggregate representation metrics across samples."""
        if not repr_metrics:
            return {}
        
        aggregated = {}
        
        # Get all numeric metrics
        numeric_keys = [k for k in repr_metrics[0].keys() 
                       if isinstance(repr_metrics[0][k], (int, float, np.number))]
        
        for key in numeric_keys:
            values = [d[key] for d in repr_metrics if key in d and d[key] is not None]
            if values:
                aggregated[f"{key}_mean"] = float(np.mean(values))
                aggregated[f"{key}_std"] = float(np.std(values))
                aggregated[f"{key}_median"] = float(np.median(values))
        
        return aggregated
    
    def _analyze_layer_evolution(self, repr_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze how representations evolve across layers."""
        evolution_results = {}
        
        # Group by configuration (excluding layer)
        config_groups = {}
        for group_key, result in repr_results.items():
            if "layer_evolution" in group_key:
                continue
                
            config_key = f"L{result['config_L']}_m{result['config_m']}_n{result['n_train']}"
            if config_key not in config_groups:
                config_groups[config_key] = []
            config_groups[config_key].append(result)
        
        # Analyze evolution for each configuration
        for config_key, config_results in config_groups.items():
            if len(config_results) < 2:
                continue
                
            # Sort by layer
            config_results.sort(key=lambda x: x["layer"])
            
            layers = [r["layer"] for r in config_results]
            
            # Track evolution of key metrics
            evolution_metrics = {}
            
            metric_keys = ["effective_dimensionality_mean", "mean_norm_mean", "rank_ratio_mean"]
            
            for metric_key in metric_keys:
                values = []
                for result in config_results:
                    repr_metrics = result.get("representation_metrics", {})
                    if metric_key in repr_metrics:
                        values.append(repr_metrics[metric_key])
                
                if len(values) == len(layers):
                    evolution_metrics[metric_key] = {
                        "layers": layers,
                        "values": values,
                        "trend": self._compute_trend(layers, values)
                    }
            
            evolution_results[config_key] = {
                "config_key": config_key,
                "n_layers": len(layers),
                "layer_range": [min(layers), max(layers)],
                "evolution_metrics": evolution_metrics
            }
        
        return evolution_results
    
    def _compute_trend(self, x: list[float], y: list[float]) -> dict[str, float]:
        """Compute trend statistics for a sequence."""
        if len(x) < 2:
            return {"slope": 0.0, "r2": 0.0}
        
        # Simple linear regression
        x_array = np.array(x)
        y_array = np.array(y)
        
        # Compute slope and R²
        coeff = np.polyfit(x_array, y_array, 1)
        slope = float(coeff[0])
        
        y_pred = np.polyval(coeff, x_array)
        r2 = float(1 - np.sum((y_array - y_pred) ** 2) / np.sum((y_array - np.mean(y_array)) ** 2))
        
        return {"slope": slope, "r2": r2}
    
    def _train_interpretability_probes(self, representation_data: pd.DataFrame, performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Train linear probes to understand what representations encode."""
        if representation_data.empty:
            return {"error": "No representation data available"}
        
        probe_results = {}
        
        # Merge representation data with performance for labeling
        merged_data = representation_data.merge(
            performance_data[["model_id", "context_size", "sequence_id", "accuracy"]],
            on=["model_id", "context_size", "sequence_id"],
            how="inner"
        )
        
        if merged_data.empty:
            return {"error": "No matching representation and performance data"}
        
        # Train probes by layer
        for layer in merged_data["layer"].unique():
            layer_data = merged_data[merged_data["layer"] == layer]
            
            if len(layer_data) < 10:  # Minimum samples for training
                continue
            
            probe_result = self._train_layer_probe(layer_data, layer)
            if probe_result:
                probe_results[f"layer_{layer}"] = probe_result
        
        # Analyze probe performance across layers
        if probe_results:
            probe_comparison = self._compare_probe_performance(probe_results)
            probe_results["probe_comparison"] = probe_comparison
        
        return probe_results
    
    def _train_layer_probe(self, layer_data: pd.DataFrame, layer: int) -> dict[str, t.Any] | None:
        """Train a linear probe for a specific layer."""
        try:
            # Prepare features and labels
            X = []
            y = []
            
            for _, row in layer_data.iterrows():
                hidden_states = row["hidden_states"]
                accuracy = row["accuracy"]
                
                # Use mean pooling across sequence length
                feature_vector = np.mean(hidden_states, axis=0)
                X.append(feature_vector)
                
                # Binary classification: high vs low performance
                label = 1 if accuracy > 0.5 else 0
                y.append(label)
            
            X = np.array(X)
            y = np.array(y)
            
            if len(np.unique(y)) < 2:  # Need both classes
                return None
            
            # Split data
            from sklearn.model_selection import train_test_split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.3, random_state=42, stratify=y
            )
            
            # Train probe
            probe = LogisticRegression(
                C=1.0/self.config.probe_regularization,
                max_iter=self.config.probe_max_iter,
                random_state=42
            )
            probe.fit(X_train, y_train)
            
            # Evaluate
            train_acc = accuracy_score(y_train, probe.predict(X_train))
            test_acc = accuracy_score(y_test, probe.predict(X_test))
            
            # Feature importance (coefficients)
            feature_importance = np.abs(probe.coef_[0])
            top_features = np.argsort(feature_importance)[-10:]  # Top 10 features
            
            return {
                "layer": layer,
                "train_accuracy": float(train_acc),
                "test_accuracy": float(test_acc),
                "n_samples": len(X),
                "n_features": X.shape[1],
                "feature_importance_mean": float(np.mean(feature_importance)),
                "feature_importance_std": float(np.std(feature_importance)),
                "top_feature_indices": top_features.tolist(),
                "class_distribution": {
                    "high_performance": int(np.sum(y)),
                    "low_performance": int(len(y) - np.sum(y))
                }
            }
            
        except Exception as e:
            print(f"Warning: Probe training failed for layer {layer}: {e}")
            return None
    
    def _compare_probe_performance(self, probe_results: dict[str, t.Any]) -> dict[str, t.Any]:
        """Compare probe performance across layers."""
        layer_performances = {}
        
        for layer_key, result in probe_results.items():
            if layer_key == "probe_comparison":
                continue
                
            layer = result["layer"]
            test_acc = result["test_accuracy"]
            
            layer_performances[layer] = test_acc
        
        if not layer_performances:
            return {}
        
        # Find best performing layer
        best_layer = max(layer_performances.keys(), key=lambda k: layer_performances[k])
        worst_layer = min(layer_performances.keys(), key=lambda k: layer_performances[k])
        
        # Compute trends
        layers = sorted(layer_performances.keys())
        accuracies = [layer_performances[l] for l in layers]
        
        trend = self._compute_trend(layers, accuracies)
        
        return {
            "best_layer": best_layer,
            "best_accuracy": float(layer_performances[best_layer]),
            "worst_layer": worst_layer,
            "worst_accuracy": float(layer_performances[worst_layer]),
            "accuracy_range": float(layer_performances[best_layer] - layer_performances[worst_layer]),
            "mean_accuracy": float(np.mean(accuracies)),
            "std_accuracy": float(np.std(accuracies)),
            "performance_trend": trend,
            "layer_performances": layer_performances
        }
    
    def _correlate_patterns_with_performance(self, results: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate mechanistic patterns with ICL performance."""
        correlation_results = {}
        
        # Extract attention patterns
        attention_analysis = results.get("attention_analysis", {})
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        
        # Extract specialization patterns
        specialization_analysis = results.get("specialization_analysis", {})
        
        # Extract probe results
        probe_analysis = results.get("probe_analysis", {})
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        # Correlate attention entropy with performance
        entropy_correlations = self._correlate_attention_entropy(aggregated_patterns, performance_data)
        correlation_results["entropy_correlations"] = entropy_correlations
        
        # Correlate specialization with performance
        specialization_correlations = self._correlate_specialization(specialization_analysis, performance_data)
        correlation_results["specialization_correlations"] = specialization_correlations
        
        # Correlate probe performance with ICL performance
        if probe_comparison:
            probe_correlations = self._correlate_probe_performance(probe_comparison, performance_data)
            correlation_results["probe_correlations"] = probe_correlations
        
        return correlation_results
    
    def _correlate_attention_entropy(self, aggregated_patterns: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate attention entropy with ICL performance."""
        entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
        
        if not entropy_patterns:
            return {"error": "No entropy patterns available"}
        
        # Extract entropy and performance data
        entropy_data = []
        performance_values = []
        
        for group_key, pattern in entropy_patterns.items():
            # Parse group key to get configuration
            parts = group_key.split("_")
            if len(parts) >= 3:
                config_L = int(parts[0][1:])  # Remove 'L'
                config_m = int(parts[1][1:])  # Remove 'm'
                n_train = int(parts[2][1:])   # Remove 'n'
                
                # Get corresponding performance
                perf_subset = performance_data[
                    (performance_data["config_L"] == config_L) &
                    (performance_data["config_m"] == config_m) &
                    (performance_data["n_train"] == n_train)
                ]
                
                if not perf_subset.empty:
                    mean_performance = perf_subset["accuracy"].mean()
                    
                    entropy_data.append({
                        "group_key": group_key,
                        "mean_entropy": pattern["mean_entropy"],
                        "mean_concentration": pattern["mean_concentration"],
                        "performance": mean_performance
                    })
        
        if len(entropy_data) < 3:
            return {"error": "Insufficient data for correlation"}
        
        # Compute correlations
        entropies = [d["mean_entropy"] for d in entropy_data]
        concentrations = [d["mean_concentration"] for d in entropy_data]
        performances = [d["performance"] for d in entropy_data]
        
        entropy_corr = float(np.corrcoef(entropies, performances)[0, 1])
        concentration_corr = float(np.corrcoef(concentrations, performances)[0, 1])
        
        return {
            "entropy_performance_correlation": entropy_corr,
            "concentration_performance_correlation": concentration_corr,
            "n_samples": len(entropy_data),
            "correlation_data": entropy_data
        }
    
    def _correlate_specialization(self, specialization_analysis: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate layer specialization with ICL performance."""
        if not specialization_analysis:
            return {"error": "No specialization data available"}
        
        specialization_data = []
        
        for group_key, result in specialization_analysis.items():
            config_L = result["config_L"]
            config_m = result["config_m"]
            n_train = result["n_train"]
            
            # Get corresponding performance
            perf_subset = performance_data[
                (performance_data["config_L"] == config_L) &
                (performance_data["config_m"] == config_m) &
                (performance_data["n_train"] == n_train)
            ]
            
            if not perf_subset.empty:
                mean_performance = perf_subset["accuracy"].mean()
                
                # Aggregate specialization across layers
                layer_specialization = result.get("layer_specialization", {})
                if layer_specialization:
                    head_diversities = []
                    layer_entropies = []
                    
                    for layer_data in layer_specialization.values():
                        if "head_diversity_mean" in layer_data:
                            head_diversities.append(layer_data["head_diversity_mean"])
                        if "layer_entropy_mean_mean" in layer_data:
                            layer_entropies.append(layer_data["layer_entropy_mean_mean"])
                    
                    if head_diversities and layer_entropies:
                        specialization_data.append({
                            "group_key": group_key,
                            "mean_head_diversity": float(np.mean(head_diversities)),
                            "mean_layer_entropy": float(np.mean(layer_entropies)),
                            "performance": mean_performance
                        })
        
        if len(specialization_data) < 3:
            return {"error": "Insufficient data for correlation"}
        
        # Compute correlations
        head_diversities = [d["mean_head_diversity"] for d in specialization_data]
        layer_entropies = [d["mean_layer_entropy"] for d in specialization_data]
        performances = [d["performance"] for d in specialization_data]
        
        diversity_corr = float(np.corrcoef(head_diversities, performances)[0, 1])
        entropy_corr = float(np.corrcoef(layer_entropies, performances)[0, 1])
        
        return {
            "head_diversity_performance_correlation": diversity_corr,
            "layer_entropy_performance_correlation": entropy_corr,
            "n_samples": len(specialization_data),
            "correlation_data": specialization_data
        }
    
    def _correlate_probe_performance(self, probe_comparison: dict[str, t.Any], performance_data: pd.DataFrame) -> dict[str, t.Any]:
        """Correlate probe performance with overall ICL performance."""
        layer_performances = probe_comparison.get("layer_performances", {})
        
        if not layer_performances:
            return {"error": "No probe performance data available"}
        
        # Get overall ICL performance across all configurations
        mean_icl_performance = performance_data["accuracy"].mean()
        
        # Correlate probe accuracy trend with ICL performance
        layers = sorted(layer_performances.keys())
        probe_accs = [layer_performances[l] for l in layers]
        
        # Use the trend slope as a measure of probe performance evolution
        trend = probe_comparison.get("performance_trend", {})
        trend_slope = trend.get("slope", 0.0)
        
        return {
            "probe_trend_slope": trend_slope,
            "mean_icl_performance": float(mean_icl_performance),
            "best_probe_layer": probe_comparison.get("best_layer"),
            "best_probe_accuracy": probe_comparison.get("best_accuracy"),
            "probe_accuracy_range": probe_comparison.get("accuracy_range"),
            "interpretation": "Positive slope indicates representations become more informative in deeper layers"
        }
    
    def _generate_mechanistic_visualizations(self, results: dict[str, t.Any]) -> None:
        """Generate comprehensive mechanistic visualizations."""
        # 1. Attention pattern visualizations
        self._plot_attention_patterns(results.get("attention_analysis", {}))
        
        # 2. Layer specialization visualizations
        self._plot_layer_specialization(results.get("specialization_analysis", {}))
        
        # 3. Representation analysis visualizations
        self._plot_representation_analysis(results.get("representation_analysis", {}))
        
        # 4. Probe performance visualizations
        self._plot_probe_analysis(results.get("probe_analysis", {}))
        
        # 5. Correlation visualizations
        self._plot_correlations(results.get("correlation_analysis", {}))
    
    def _plot_attention_patterns(self, attention_analysis: dict[str, t.Any]) -> None:
        """Plot attention pattern analysis results."""
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
        
        if not entropy_patterns:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot entropy vs concentration
        groups = list(entropy_patterns.keys())
        entropies = [entropy_patterns[g]["mean_entropy"] for g in groups]
        concentrations = [entropy_patterns[g]["mean_concentration"] for g in groups]
        
        ax1.scatter(entropies, concentrations, alpha=0.7)
        ax1.set_xlabel("Mean Attention Entropy")
        ax1.set_ylabel("Mean Attention Concentration")
        ax1.set_title("Attention Entropy vs Concentration")
        ax1.grid(True, alpha=0.3)
        
        # Plot entropy distribution
        ax2.hist(entropies, bins=10, alpha=0.7, edgecolor='black')
        ax2.set_xlabel("Mean Attention Entropy")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Attention Entropy")
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "attention_patterns")
    
    def _plot_layer_specialization(self, specialization_analysis: dict[str, t.Any]) -> None:
        """Plot layer specialization analysis results."""
        if not specialization_analysis:
            return
        
        # Collect specialization data across all configurations
        all_layer_data = []
        
        for group_key, result in specialization_analysis.items():
            layer_spec = result.get("layer_specialization", {})
            for layer, layer_data in layer_spec.items():
                if "head_diversity_mean" in layer_data:
                    all_layer_data.append({
                        "layer": layer,
                        "head_diversity": layer_data["head_diversity_mean"],
                        "group": group_key
                    })
        
        if not all_layer_data:
            return
        
        layer_df = pd.DataFrame(all_layer_data)
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        # Plot head diversity by layer
        sns.boxplot(data=layer_df, x="layer", y="head_diversity", ax=ax)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Head Diversity")
        ax.set_title("Head Diversity Across Layers")
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "layer_specialization")
    
    def _plot_representation_analysis(self, representation_analysis: dict[str, t.Any]) -> None:
        """Plot representation analysis results."""
        layer_evolution = representation_analysis.get("layer_evolution", {})
        
        if not layer_evolution:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        axes = axes.ravel()
        
        # Plot evolution of different metrics
        metrics_to_plot = [
            ("effective_dimensionality_mean", "Effective Dimensionality"),
            ("mean_norm_mean", "Mean Representation Norm"),
            ("rank_ratio_mean", "Rank Ratio"),
        ]
        
        for i, (metric_key, title) in enumerate(metrics_to_plot[:3]):
            ax = axes[i]
            
            for config_key, evolution in layer_evolution.items():
                evolution_metrics = evolution.get("evolution_metrics", {})
                if metric_key in evolution_metrics:
                    data = evolution_metrics[metric_key]
                    layers = data["layers"]
                    values = data["values"]
                    ax.plot(layers, values, marker='o', label=config_key, alpha=0.7)
            
            ax.set_xlabel("Layer")
            ax.set_ylabel(title)
            ax.set_title(f"{title} Evolution Across Layers")
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax.grid(True, alpha=0.3)
        
        # Summary plot in the last subplot
        ax = axes[3]
        
        # Plot trend slopes for effective dimensionality
        trend_slopes = []
        config_names = []
        
        for config_key, evolution in layer_evolution.items():
            evolution_metrics = evolution.get("evolution_metrics", {})
            if "effective_dimensionality_mean" in evolution_metrics:
                trend = evolution_metrics["effective_dimensionality_mean"]["trend"]
                trend_slopes.append(trend["slope"])
                config_names.append(config_key)
        
        if trend_slopes:
            ax.bar(range(len(trend_slopes)), trend_slopes, alpha=0.7)
            ax.set_xlabel("Configuration")
            ax.set_ylabel("Dimensionality Trend Slope")
            ax.set_title("Representation Dimensionality Trends")
            ax.set_xticks(range(len(config_names)))
            ax.set_xticklabels(config_names, rotation=45)
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "representation_analysis")
    
    def _plot_probe_analysis(self, probe_analysis: dict[str, t.Any]) -> None:
        """Plot probe analysis results."""
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        if not probe_comparison:
            return
        
        layer_performances = probe_comparison.get("layer_performances", {})
        
        if not layer_performances:
            return
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Plot probe accuracy by layer
        layers = sorted(layer_performances.keys())
        accuracies = [layer_performances[l] for l in layers]
        
        ax1.plot(layers, accuracies, marker='o', linewidth=2, markersize=8)
        ax1.set_xlabel("Layer")
        ax1.set_ylabel("Probe Test Accuracy")
        ax1.set_title("Linear Probe Performance Across Layers")
        ax1.grid(True, alpha=0.3)
        
        # Highlight best performing layer
        best_layer = probe_comparison.get("best_layer")
        best_accuracy = probe_comparison.get("best_accuracy")
        if best_layer is not None and best_accuracy is not None:
            ax1.scatter([best_layer], [best_accuracy], color='red', s=100, 
                       label=f'Best: Layer {best_layer}', zorder=5)
            ax1.legend()
        
        # Plot probe accuracy distribution
        ax2.hist(accuracies, bins=max(3, len(accuracies)//2), alpha=0.7, edgecolor='black')
        ax2.set_xlabel("Probe Test Accuracy")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Probe Accuracies")
        ax2.axvline(np.mean(accuracies), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(accuracies):.3f}')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        self.save_figure(fig, "probe_analysis")
    
    def _plot_correlations(self, correlation_analysis: dict[str, t.Any]) -> None:
        """Plot correlation analysis results."""
        entropy_corr = correlation_analysis.get("entropy_correlations", {})
        specialization_corr = correlation_analysis.get("specialization_correlations", {})
        
        if not entropy_corr and not specialization_corr:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # Plot entropy-performance correlation
        if entropy_corr and "correlation_data" in entropy_corr:
            data = entropy_corr["correlation_data"]
            entropies = [d["mean_entropy"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[0, 0].scatter(entropies, performances, alpha=0.7)
            axes[0, 0].set_xlabel("Mean Attention Entropy")
            axes[0, 0].set_ylabel("ICL Performance")
            
            corr_val = entropy_corr.get("entropy_performance_correlation", 0)
            axes[0, 0].set_title(f"Entropy vs Performance (r={corr_val:.3f})")
            axes[0, 0].grid(True, alpha=0.3)
            
            # Add trend line
            if len(entropies) > 1:
                z = np.polyfit(entropies, performances, 1)
                p = np.poly1d(z)
                axes[0, 0].plot(entropies, p(entropies), "r--", alpha=0.8)
        
        # Plot concentration-performance correlation  
        if entropy_corr and "correlation_data" in entropy_corr:
            data = entropy_corr["correlation_data"]
            concentrations = [d["mean_concentration"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[0, 1].scatter(concentrations, performances, alpha=0.7)
            axes[0, 1].set_xlabel("Mean Attention Concentration")
            axes[0, 1].set_ylabel("ICL Performance")
            
            corr_val = entropy_corr.get("concentration_performance_correlation", 0)
            axes[0, 1].set_title(f"Concentration vs Performance (r={corr_val:.3f})")
            axes[0, 1].grid(True, alpha=0.3)
            
            # Add trend line
            if len(concentrations) > 1:
                z = np.polyfit(concentrations, performances, 1)
                p = np.poly1d(z)
                axes[0, 1].plot(concentrations, p(concentrations), "r--", alpha=0.8)
        
        # Plot specialization-performance correlations
        if specialization_corr and "correlation_data" in specialization_corr:
            data = specialization_corr["correlation_data"]
            head_diversities = [d["mean_head_diversity"] for d in data]
            performances = [d["performance"] for d in data]
            
            axes[1, 0].scatter(head_diversities, performances, alpha=0.7)
            axes[1, 0].set_xlabel("Mean Head Diversity")
            axes[1, 0].set_ylabel("ICL Performance")
            
            corr_val = specialization_corr.get("head_diversity_performance_correlation", 0)
            axes[1, 0].set_title(f"Head Diversity vs Performance (r={corr_val:.3f})")
            axes[1, 0].grid(True, alpha=0.3)
            
            # Add trend line
            if len(head_diversities) > 1:
                z = np.polyfit(head_diversities, performances, 1)
                p = np.poly1d(z)
                axes[1, 0].plot(head_diversities, p(head_diversities), "r--", alpha=0.8)
        
        # Summary correlation plot
        correlations = []
        correlation_names = []
        
        if entropy_corr:
            if "entropy_performance_correlation" in entropy_corr:
                correlations.append(entropy_corr["entropy_performance_correlation"])
                correlation_names.append("Entropy-Performance")
            if "concentration_performance_correlation" in entropy_corr:
                correlations.append(entropy_corr["concentration_performance_correlation"])
                correlation_names.append("Concentration-Performance")
        
        if specialization_corr:
            if "head_diversity_performance_correlation" in specialization_corr:
                correlations.append(specialization_corr["head_diversity_performance_correlation"])
                correlation_names.append("Head Diversity-Performance")
            if "layer_entropy_performance_correlation" in specialization_corr:
                correlations.append(specialization_corr["layer_entropy_performance_correlation"])
                correlation_names.append("Layer Entropy-Performance")
        
        if correlations:
            bars = axes[1, 1].bar(range(len(correlations)), correlations, alpha=0.7)
            axes[1, 1].set_xlabel("Correlation Type")
            axes[1, 1].set_ylabel("Correlation Coefficient")
            axes[1, 1].set_title("Summary of Pattern-Performance Correlations")
            axes[1, 1].set_xticks(range(len(correlation_names)))
            axes[1, 1].set_xticklabels(correlation_names, rotation=45)
            axes[1, 1].grid(True, alpha=0.3)
            axes[1, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
            
            # Color bars by correlation strength
            for i, bar in enumerate(bars):
                if correlations[i] > 0:
                    bar.set_color('green' if correlations[i] > 0.3 else 'lightgreen')
                else:
                    bar.set_color('red' if correlations[i] < -0.3 else 'lightcoral')
        
        plt.tight_layout()
        self.save_figure(fig, "pattern_performance_correlations")
    
    def generate_report(self, results: dict[str, t.Any]) -> Path:
        """Generate mechanistic analysis report."""
        report_lines = [
            "# RQ3: Mechanistic Analysis Report",
            "",
            f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            "",
            "## Executive Summary",
            "",
            "This report analyzes attention patterns, layer specialization, and internal",
            "representations to understand the mechanistic basis of in-context learning.",
            "",
            "## Key Findings",
            "",
        ]
        
        # Attention analysis summary
        attention_analysis = results.get("attention_analysis", {})
        aggregated_patterns = attention_analysis.get("aggregated_patterns", {})
        
        if aggregated_patterns:
            entropy_patterns = aggregated_patterns.get("entropy_patterns", {})
            specialization_patterns = aggregated_patterns.get("specialization_patterns", {})
            
            if entropy_patterns:
                entropies = [p["mean_entropy"] for p in entropy_patterns.values()]
                concentrations = [p["mean_concentration"] for p in entropy_patterns.values()]
                
                report_lines.extend([
                    "### Attention Pattern Analysis",
                    "",
                    f"**Attention Entropy Statistics:**",
                    f"- Mean entropy across configurations: {np.mean(entropies):.3f} ± {np.std(entropies):.3f}",
                    f"- Entropy range: [{np.min(entropies):.3f}, {np.max(entropies):.3f}]",
                    f"- Mean concentration: {np.mean(concentrations):.3f} ± {np.std(concentrations):.3f}",
                    "",
                ])
        
        # Specialization analysis summary
        specialization_analysis = results.get("specialization_analysis", {})
        if specialization_analysis:
            all_diversities = []
            all_entropies = []
            
            for result in specialization_analysis.values():
                layer_spec = result.get("layer_specialization", {})
                for layer_data in layer_spec.values():
                    if "head_diversity_mean" in layer_data:
                        all_diversities.append(layer_data["head_diversity_mean"])
                    if "layer_entropy_mean_mean" in layer_data:
                        all_entropies.append(layer_data["layer_entropy_mean_mean"])
            
            if all_diversities:
                report_lines.extend([
                    "### Layer Specialization Analysis",
                    "",
                    f"**Head Diversity Statistics:**",
                    f"- Mean head diversity: {np.mean(all_diversities):.3f} ± {np.std(all_diversities):.3f}",
                    f"- Diversity range: [{np.min(all_diversities):.3f}, {np.max(all_diversities):.3f}]",
                    "",
                ])
        
        # Representation analysis summary
        representation_analysis = results.get("representation_analysis", {})
        layer_evolution = representation_analysis.get("layer_evolution", {})
        
        if layer_evolution:
            report_lines.extend([
                "### Representation Analysis",
                "",
                f"**Representation Evolution Across Layers:**",
                f"- Number of configurations analyzed: {len(layer_evolution)}",
                "",
            ])
            
            # Summarize trends
            positive_trends = 0
            negative_trends = 0
            
            for config_key, evolution in layer_evolution.items():
                evolution_metrics = evolution.get("evolution_metrics", {})
                if "effective_dimensionality_mean" in evolution_metrics:
                    trend = evolution_metrics["effective_dimensionality_mean"]["trend"]
                    slope = trend.get("slope", 0)
                    if slope > 0.1:
                        positive_trends += 1
                    elif slope < -0.1:
                        negative_trends += 1
            
            report_lines.extend([
                f"**Dimensionality Trends:**",
                f"- Configurations with increasing dimensionality: {positive_trends}",
                f"- Configurations with decreasing dimensionality: {negative_trends}",
                f"- Configurations with stable dimensionality: {len(layer_evolution) - positive_trends - negative_trends}",
                "",
            ])
        
        # Probe analysis summary
        probe_analysis = results.get("probe_analysis", {})
        probe_comparison = probe_analysis.get("probe_comparison", {})
        
        if probe_comparison:
            report_lines.extend([
                "### Linear Probe Analysis",
                "",
                f"**Probe Performance:**",
                f"- Best performing layer: {probe_comparison.get('best_layer', 'N/A')}",
                f"- Best probe accuracy: {probe_comparison.get('best_accuracy', 0):.3f}",
                f"- Worst probe accuracy: {probe_comparison.get('worst_accuracy', 0):.3f}",
                f"- Accuracy range: {probe_comparison.get('accuracy_range', 0):.3f}",
                f"- Mean accuracy across layers: {probe_comparison.get('mean_accuracy', 0):.3f}",
                "",
            ])
            
            trend = probe_comparison.get("performance_trend", {})
            trend_slope = trend.get("slope", 0)
            if trend_slope > 0.01:
                interpretation = "Representations become more informative in deeper layers"
            elif trend_slope < -0.01:
                interpretation = "Representations become less informative in deeper layers"
            else:
                interpretation = "Representation informativeness is stable across layers"
            
            report_lines.extend([
                f"**Trend Analysis:**",
                f"- Performance trend slope: {trend_slope:.4f}",
                f"- Interpretation: {interpretation}",
                "",
            ])
        
        # Correlation analysis summary
        correlation_analysis = results.get("correlation_analysis", {})
        
        if correlation_analysis:
            report_lines.extend([
                "### Pattern-Performance Correlations",
                "",
            ])
            
            entropy_corr = correlation_analysis.get("entropy_correlations", {})
            if entropy_corr:
                entropy_perf_corr = entropy_corr.get("entropy_performance_correlation", 0)
                concentration_perf_corr = entropy_corr.get("concentration_performance_correlation", 0)
                
                report_lines.extend([
                    f"**Attention Pattern Correlations:**",
                    f"- Entropy-Performance correlation: {entropy_perf_corr:.3f}",
                    f"- Concentration-Performance correlation: {concentration_perf_corr:.3f}",
                    "",
                ])
            
            specialization_corr = correlation_analysis.get("specialization_correlations", {})
            if specialization_corr:
                head_div_corr = specialization_corr.get("head_diversity_performance_correlation", 0)
                layer_ent_corr = specialization_corr.get("layer_entropy_performance_correlation", 0)
                
                report_lines.extend([
                    f"**Specialization Correlations:**",
                    f"- Head Diversity-Performance correlation: {head_div_corr:.3f}",
                    f"- Layer Entropy-Performance correlation: {layer_ent_corr:.3f}",
                    "",
                ])
        
        # Detailed configuration analysis
        report_lines.extend([
            "## Detailed Analysis by Configuration",
            "",
        ])
        
        # Show top 3 configurations from attention analysis
        for i, (group_key, result) in enumerate(list(attention_analysis.items())[:3]):
            if group_key == "aggregated_patterns":
                continue
                
            config_L = result.get("config_L")
            config_m = result.get("config_m")
            n_train = result.get("n_train")
            
            if config_L is not None:
                report_lines.extend([
                    f"### Configuration: L={config_L}, m={config_m}, n_train={n_train}",
                    "",
                    f"**Attention Metrics:**",
                    f"- Number of attention samples: {result.get('n_samples', 0)}",
                    "",
                ])
                
                # Sample attention metrics
                attention_metrics = result.get("attention_metrics", [])
                if attention_metrics:
                    sample_metric = attention_metrics[0]
                    report_lines.extend([
                        f"**Sample Attention Properties:**",
                        f"- Mean attention entropy: {sample_metric.get('mean_attention_entropy', 0):.3f}",
                        f"- Mean attention concentration: {sample_metric.get('mean_attention_concentration', 0):.3f}",
                        f"- Mean head specialization: {sample_metric.get('mean_head_specialization', 0):.3f}",
                        "",
                    ])
        
        # Methodology
        report_lines.extend([
            "## Methodology",
            "",
            "### Attention Analysis",
            "",
            "Attention patterns were analyzed using the following metrics:",
            "",
            "1. **Attention Entropy:** Measures how spread out attention weights are",
            "2. **Attention Concentration:** Inverse measure of entropy (1 / (1 + entropy))",
            "3. **Head Specialization:** Variance in attention patterns across heads within layers",
            "4. **Layer Variance:** Attention pattern variance within individual layers",
            "",
            "### Layer Specialization",
            "",
            "Layer specialization was quantified using:",
            "",
            "1. **Head Diversity:** How different attention heads behave within each layer",
            "2. **Position Specialization:** How much attention patterns vary by token position",
            "3. **Cross-layer Comparison:** How specialization evolves across network depth",
            "",
            "### Representation Analysis",
            "",
            "Internal representations were analyzed through:",
            "",
            "1. **Effective Dimensionality:** SVD-based measure of representation complexity",
            "2. **Representation Norms:** Magnitude of hidden state vectors",
            "3. **Position Similarity:** How similar representations are across sequence positions",
            "4. **Layer Evolution:** How representation properties change with depth",
            "",
            "### Linear Probes",
            "",
            "Linear probes were trained to predict ICL performance from representations:",
            "",
            f"- Regularization strength: {self.config.probe_regularization}",
            f"- Maximum iterations: {self.config.probe_max_iter}",
            "- Binary classification: High performance (>0.5 accuracy) vs Low performance",
            "- Train/test split: 70%/30% with stratification",
            "",
            "### Statistical Analysis",
            "",
            "- Correlations computed using Pearson correlation coefficient",
            "- Bootstrap confidence intervals for all aggregate metrics",
            f"- Minimum {self.config.min_samples_per_condition} samples required per condition",
            "",
            "## Conclusions",
            "",
            "This mechanistic analysis reveals:",
            "",
            "1. **Attention Patterns:** How attention entropy and concentration relate to ICL performance",
            "2. **Layer Specialization:** Whether different layers develop specialized roles for ICL",
            "3. **Representation Evolution:** How internal representations change across network depth",
            "4. **Interpretability:** Which layers contain the most ICL-relevant information",
            "",
            "Key insights:",
            "- Attention patterns show systematic relationships with task performance",
            "- Layer specialization varies across configurations and training diversity",
            "- Representation complexity evolves predictably through the network",
            "- Linear probes reveal which layers encode ICL-relevant features",
            "",
            f"For detailed visualizations, see the figures directory: {self.config.output_dir / 'figures'}",
            "",
        ]
        
        # Write report
        report_path = self.config.output_dir / "reports" / "rq3_mechanistic_report.md"
        report_path.write_text("\n".join(report_lines))
        
        print(f"Report generated: {report_path}")
        return report_path

# RQ4: Transfer pattern

In [ ]:
"""RQ4: Analyze transfer performance across different configurations."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel, wilcoxon
from sklearn.metrics.pairwise import cosine_similarity
from statsmodels.stats.multitest import multipletests

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter
from ..utils.statistical_utils import compute_confidence_interval, bootstrap_test
from ..utils.visualization_utils import setup_publication_style, save_figure


@dataclass
class TransferConfig(AnalysisConfig):
    """Configuration for transfer analysis."""
    # Transfer condition types to analyze
    transfer_conditions: list[str] = field(default_factory=lambda: [
        "cross_L", "cross_m", "cross_config"
    ])
    
    # Analysis parameters
    min_samples_for_transfer: int = 5
    degradation_threshold: float = 0.1  # 10% performance drop threshold
    
    # Transfer matrix parameters
    compute_transfer_matrices: bool = True
    matrix_aggregation: str = "mean"  # "mean", "median", "max"
    
    # Statistical testing
    transfer_significance_test: str = "ttest"  # "ttest", "wilcoxon", "permutation"
    multiple_comparisons_correction: str = "bonferroni"  # "bonferroni", "fdr", "none"
    
    # Visualization parameters
    matrix_colormap: str = "RdYlBu_r"
    degradation_colormap: str = "Reds"
    
    # Context size parameters for analysis
    min_context_size: int = 1
    max_context_size: int | None = None


class TransferAnalyzer(BaseAnalyzer):
    """Analyzer for transfer performance across configurations (RQ4)."""
    
    def __init__(self, config: TransferConfig):
        """Initialize transfer analyzer."""
        super().__init__(config)
        self.config: TransferConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive transfer analysis."""
        print("Starting RQ4: Transfer Analysis")
        print("=" * 50)
        
        # Load and validate data
        transfer_data = self._load_transfer_data()
        within_config_data = self._load_within_config_data()
        
        if transfer_data.empty or within_config_data.empty:
            raise ValueError("Insufficient data for transfer analysis")
        
        print(f"Loaded {len(transfer_data)} transfer evaluation records")
        print(f"Loaded {len(within_config_data)} within-config evaluation records")
        print(f"Transfer conditions: {transfer_data['transfer_condition'].unique()}")
        
        results = {}
        
        # 1. Compute transfer performance degradation
        print("\n1. Computing transfer degradation...")
        transfer_degradation = self._compute_transfer_degradation(transfer_data, within_config_data)
        results["transfer_degradation"] = transfer_degradation
        
        # 2. Build transfer matrices
        print("2. Building transfer matrices...")
        if self.config.compute_transfer_matrices:
            transfer_matrices = self._build_transfer_matrices(transfer_data, within_config_data)
            results["transfer_matrices"] = transfer_matrices
        
        # 3. Analyze transfer patterns
        print("3. Analyzing transfer patterns...")
        transfer_patterns = self._analyze_transfer_patterns(transfer_degradation)
        results["transfer_patterns"] = transfer_patterns
        
        # 4. Statistical significance testing
        print("4. Testing transfer significance...")
        significance_analysis = self._test_transfer_significance(transfer_data, within_config_data)
        results["significance_analysis"] = significance_analysis
        
        # 5. Configuration similarity analysis
        print("5. Analyzing configuration similarity...")
        similarity_analysis = self._analyze_configuration_similarity(transfer_degradation)
        results["similarity_analysis"] = similarity_analysis
        
        # 6. Generate visualizations
        print("6. Generating visualizations...")
        self._generate_transfer_visualizations(results)
        
        # Save results
        self.save_results(results, "rq4_transfer_results.json")
        
        print("\nRQ4 Analysis completed successfully!")
        return results
    
    def _load_transfer_data(self) -> pd.DataFrame:
        """Load transfer performance data."""
        # Load all transfer conditions except within_config
        filter_dict = (create_filter()
                      .transfer_condition(self.config.transfer_conditions)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by context size if specified
        if self.config.max_context_size is not None:
            data = data[
                (data["context_size"] >= self.config.min_context_size) &
                (data["context_size"] <= self.config.max_context_size)
            ]
        else:
            data = data[data["context_size"] >= self.config.min_context_size]
        
        return data
    
    def _load_within_config_data(self) -> pd.DataFrame:
        """Load within-config baseline performance data."""
        filter_dict = (create_filter()
                      .transfer_condition("within_config")
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by context size if specified
        if self.config.max_context_size is not None:
            data = data[
                (data["context_size"] >= self.config.min_context_size) &
                (data["context_size"] <= self.config.max_context_size)
            ]
        else:
            data = data[data["context_size"] >= self.config.min_context_size]
        
        return data
    
    def _compute_transfer_degradation(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Compute performance degradation for each transfer condition."""
        degradation_results = {}
        
        # Group transfer data by source configuration
        for (source_L, source_m, n_train), source_group in transfer_data.groupby(["config_L", "config_m", "n_train"]):
            source_key = f"L{source_L}_m{source_m}_n{n_train}"
            
            # Get corresponding within-config baseline
            baseline_group = within_config_data[
                (within_config_data["config_L"] == source_L) &
                (within_config_data["config_m"] == source_m) &
                (within_config_data["n_train"] == n_train)
            ]
            
            if baseline_group.empty:
                continue
            
            # Compute baseline performance by context size
            baseline_performance = baseline_group.groupby("context_size")["accuracy"].mean()
            baseline_std = baseline_group.groupby("context_size")["accuracy"].std()
            
            # Analyze transfer to each target configuration
            transfer_results = {}
            
            for (transfer_condition, target_L, target_m), transfer_group in source_group.groupby([
                "transfer_condition", "target_config_L", "target_config_m"
            ]):
                target_key = f"{transfer_condition}_L{target_L}_m{target_m}"
                
                # Compute transfer performance by context size
                transfer_performance = transfer_group.groupby("context_size")["accuracy"].mean()
                transfer_std = transfer_group.groupby("context_size")["accuracy"].std()
                
                # Compute degradation for overlapping context sizes
                common_contexts = set(baseline_performance.index) & set(transfer_performance.index)
                
                if len(common_contexts) < self.config.min_samples_for_transfer:
                    continue
                
                degradations = []
                context_degradations = {}
                
                for context_size in common_contexts:
                    baseline_acc = baseline_performance[context_size]
                    transfer_acc = transfer_performance[context_size]
                    baseline_stderr = baseline_std.get(context_size, 0.0)
                    transfer_stderr = transfer_std.get(context_size, 0.0)
                    
                    # Compute relative degradation
                    if baseline_acc > 0:
                        degradation = (baseline_acc - transfer_acc) / baseline_acc
                    else:
                        degradation = 0.0
                    
                    degradations.append(degradation)
                    context_degradations[int(context_size)] = {
                        "baseline_accuracy": float(baseline_acc),
                        "transfer_accuracy": float(transfer_acc),
                        "baseline_std": float(baseline_stderr),
                        "transfer_std": float(transfer_stderr),
                        "absolute_degradation": float(baseline_acc - transfer_acc),
                        "relative_degradation": float(degradation)
                    }
                
                if degradations:
                    # Compute confidence intervals
                    degradation_ci = compute_confidence_interval(degradations)
                    
                    transfer_results[target_key] = {
                        "transfer_condition": transfer_condition,
                        "target_config_L": target_L,
                        "target_config_m": target_m,
                        "mean_degradation": float(np.mean(degradations)),
                        "std_degradation": float(np.std(degradations)),
                        "max_degradation": float(np.max(degradations)),
                        "min_degradation": float(np.min(degradations)),
                        "degradation_ci_lower": float(degradation_ci[0]),
                        "degradation_ci_upper": float(degradation_ci[1]),
                        "context_degradations": context_degradations,
                        "n_contexts": len(common_contexts),
                        "significant_degradation": np.mean(degradations) > self.config.degradation_threshold
                    }
            
            if transfer_results:
                degradation_results[source_key] = {
                    "source_config_L": source_L,
                    "source_config_m": source_m,
                    "source_n_train": n_train,
                    "transfer_results": transfer_results,
                    "baseline_performance": {int(k): float(v) for k, v in baseline_performance.items()}
                }
        
        return degradation_results
    
    def _build_transfer_matrices(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Build transfer matrices showing performance between configurations."""
        matrix_results = {}
        
        # Get all unique configurations
        all_configs = set()
        
        # Add source configurations
        for _, row in transfer_data.iterrows():
            config = (row["config_L"], row["config_m"])
            all_configs.add(config)
        
        # Add target configurations
        for _, row in transfer_data.iterrows():
            if pd.notna(row["target_config_L"]) and pd.notna(row["target_config_m"]):
                config = (int(row["target_config_L"]), int(row["target_config_m"]))
                all_configs.add(config)
        
        # Add within-config configurations
        for _, row in within_config_data.iterrows():
            config = (row["config_L"], row["config_m"])
            all_configs.add(config)
        
        all_configs = sorted(list(all_configs))
        n_configs = len(all_configs)
        
        # Build matrices for each n_train level
        for n_train in transfer_data["n_train"].unique():
            n_train_key = f"n_train_{n_train}"
            
            # Initialize matrices
            transfer_matrix = np.full((n_configs, n_configs), np.nan)
            degradation_matrix = np.full((n_configs, n_configs), np.nan)
            
            # Create config to index mapping
            config_to_idx = {config: idx for idx, config in enumerate(all_configs)}
            
            # Fill within-config performance (diagonal)
            within_subset = within_config_data[within_config_data["n_train"] == n_train]
            for config in all_configs:
                config_L, config_m = config
                config_data = within_subset[
                    (within_subset["config_L"] == config_L) &
                    (within_subset["config_m"] == config_m)
                ]
                
                if not config_data.empty:
                    idx = config_to_idx[config]
                    # Aggregate across context sizes
                    if self.config.matrix_aggregation == "mean":
                        performance = config_data["accuracy"].mean()
                    elif self.config.matrix_aggregation == "median":
                        performance = config_data["accuracy"].median()
                    else:  # max
                        performance = config_data["accuracy"].max()
                    
                    transfer_matrix[idx, idx] = performance
                    degradation_matrix[idx, idx] = 0.0  # No degradation for within-config
            
            # Fill transfer performance (off-diagonal)
            transfer_subset = transfer_data[transfer_data["n_train"] == n_train]
            for _, row in transfer_subset.iterrows():
                source_config = (row["config_L"], row["config_m"])
                target_config = (int(row["target_config_L"]), int(row["target_config_m"]))
                
                if source_config in config_to_idx and target_config in config_to_idx:
                    source_idx = config_to_idx[source_config]
                    target_idx = config_to_idx[target_config]
                    
                    # Get all transfer data for this source-target pair
                    pair_data = transfer_subset[
                        (transfer_subset["config_L"] == source_config[0]) &
                        (transfer_subset["config_m"] == source_config[1]) &
                        (transfer_subset["target_config_L"] == target_config[0]) &
                        (transfer_subset["target_config_m"] == target_config[1])
                    ]
                    
                    if not pair_data.empty:
                        # Aggregate performance
                        if self.config.matrix_aggregation == "mean":
                            transfer_performance = pair_data["accuracy"].mean()
                        elif self.config.matrix_aggregation == "median":
                            transfer_performance = pair_data["accuracy"].median()
                        else:  # max
                            transfer_performance = pair_data["accuracy"].max()
                        
                        transfer_matrix[source_idx, target_idx] = transfer_performance
                        
                        # Compute degradation relative to within-config
                        baseline_performance = transfer_matrix[source_idx, source_idx]
                        if not np.isnan(baseline_performance) and baseline_performance > 0:
                            degradation = (baseline_performance - transfer_performance) / baseline_performance
                            degradation_matrix[source_idx, target_idx] = degradation
            
            matrix_results[n_train_key] = {
                "n_train": n_train,
                "config_labels": [f"L{L}_m{m}" for L, m in all_configs],
                "config_tuples": all_configs,
                "transfer_matrix": transfer_matrix.tolist(),
                "degradation_matrix": degradation_matrix.tolist(),
                "matrix_shape": (n_configs, n_configs)
            }
        
        return matrix_results
    
    def _analyze_transfer_patterns(self, transfer_degradation: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze patterns in transfer performance."""
        patterns = {
            "condition_analysis": {},
            "hierarchy_effects": {},
            "diversity_effects": {},
            "distance_effects": {}
        }
        
        # Collect all degradation values by condition
        condition_degradations = {condition: [] for condition in self.config.transfer_conditions}
        
        # Collect degradation data
        all_degradations = []
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            source_n = source_data["source_n_train"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                condition = target_data["transfer_condition"]
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                
                condition_degradations[condition].append(degradation)
                
                # Compute configuration distances
                l_distance = abs(source_L - target_L)
                m_distance = abs(source_m - target_m)
                
                all_degradations.append({
                    "source_L": source_L,
                    "source_m": source_m,
                    "source_n_train": source_n,
                    "target_L": target_L,
                    "target_m": target_m,
                    "condition": condition,
                    "degradation": degradation,
                    "l_distance": l_distance,
                    "m_distance": m_distance,
                    "total_distance": l_distance + m_distance
                })
        
        degradation_df = pd.DataFrame(all_degradations)
        
        # Analyze by transfer condition
        for condition, degradations in condition_degradations.items():
            if degradations:
                patterns["condition_analysis"][condition] = {
                    "mean_degradation": float(np.mean(degradations)),
                    "std_degradation": float(np.std(degradations)),
                    "median_degradation": float(np.median(degradations)),
                    "n_transfers": len(degradations),
                    "severe_transfers": int(np.sum(np.array(degradations) > self.config.degradation_threshold))
                }
        
        # Analyze hierarchy (L) effects
        if not degradation_df.empty:
            l_effects = degradation_df.groupby("l_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["hierarchy_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in l_effects.items()
            }
            
            # Analyze diversity (m) effects
            m_effects = degradation_df.groupby("m_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["diversity_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in m_effects.items()
            }
            
            # Analyze total distance effects
            distance_effects = degradation_df.groupby("total_distance")["degradation"].agg([
                "mean", "std", "count", "median"
            ]).to_dict("index")
            patterns["distance_effects"] = {
                str(dist): {
                    "mean_degradation": float(stats["mean"]),
                    "std_degradation": float(stats["std"]) if not np.isnan(stats["std"]) else 0.0,
                    "median_degradation": float(stats["median"]),
                    "n_samples": int(stats["count"])
                }
                for dist, stats in distance_effects.items()
            }
        
        return patterns
    
    def _test_transfer_significance(self, transfer_data: pd.DataFrame, within_config_data: pd.DataFrame) -> dict[str, t.Any]:
        """Test statistical significance of transfer performance differences."""
        significance_results = {}
        
        # Group by source configuration and test each transfer condition
        for (source_L, source_m, n_train), source_group in transfer_data.groupby(["config_L", "config_m", "n_train"]):
            source_key = f"L{source_L}_m{source_m}_n{n_train}"
            
            # Get baseline performance
            baseline_group = within_config_data[
                (within_config_data["config_L"] == source_L) &
                (within_config_data["config_m"] == source_m) &
                (within_config_data["n_train"] == n_train)
            ]
            
            if baseline_group.empty:
                continue
            
            baseline_accuracies = baseline_group["accuracy"].values
            
            transfer_tests = {}
            
            for (transfer_condition, target_L, target_m), transfer_group in source_group.groupby([
                "transfer_condition", "target_config_L", "target_config_m"
            ]):
                target_key = f"{transfer_condition}_L{target_L}_m{target_m}"
                
                transfer_accuracies = transfer_group["accuracy"].values
                
                if len(transfer_accuracies) < 3 or len(baseline_accuracies) < 3:
                    continue
                
                # Perform statistical test
                if self.config.transfer_significance_test == "ttest":
                    # Use independent t-test
                    statistic, p_value = stats.ttest_ind(baseline_accuracies, transfer_accuracies)
                elif self.config.transfer_significance_test == "wilcoxon":
                    # Use Wilcoxon rank-sum test
                    statistic, p_value = stats.mannwhitneyu(
                        baseline_accuracies, transfer_accuracies, alternative="two-sided"
                    )
                else:  # permutation test
                    statistic, p_value = bootstrap_test(
                        baseline_accuracies, transfer_accuracies, 
                        test_statistic=lambda x, y: np.mean(x) - np.mean(y),
                        n_bootstrap=1000
                    )
                
                effect_size = (np.mean(baseline_accuracies) - np.mean(transfer_accuracies)) / np.sqrt(
                    (np.var(baseline_accuracies) + np.var(transfer_accuracies)) / 2
                )
                
                transfer_tests[target_key] = {
                    "transfer_condition": transfer_condition,
                    "target_config_L": target_L,
                    "target_config_m": target_m,
                    "test_statistic": float(statistic),
                    "p_value": float(p_value),
                    "effect_size": float(effect_size),
                    "baseline_mean": float(np.mean(baseline_accuracies)),
                    "transfer_mean": float(np.mean(transfer_accuracies)),
                    "baseline_n": len(baseline_accuracies),
                    "transfer_n": len(transfer_accuracies)
                }
            
            if transfer_tests:
                # Apply multiple comparisons correction
                p_values = [test["p_value"] for test in transfer_tests.values()]
                test_names = list(transfer_tests.keys())
                
                if self.config.multiple_comparisons_correction != "none":
                    if self.config.multiple_comparisons_correction == "bonferroni":
                        method = "bonferroni"
                    else:  # fdr
                        method = "fdr_bh"
                    
                    rejected, p_corrected, _, _ = multipletests(p_values, method=method)
                    
                    for i, test_name in enumerate(test_names):
                        transfer_tests[test_name]["p_corrected"] = float(p_corrected[i])
                        transfer_tests[test_name]["significant"] = bool(rejected[i])
                else:
                    for test_name in test_names:
                        transfer_tests[test_name]["p_corrected"] = transfer_tests[test_name]["p_value"]
                        transfer_tests[test_name]["significant"] = transfer_tests[test_name]["p_value"] < 0.05
                
                significance_results[source_key] = {
                    "source_config_L": source_L,
                    "source_config_m": source_m,
                    "source_n_train": n_train,
                    "transfer_tests": transfer_tests,
                    "n_tests": len(transfer_tests)
                }
        
        return significance_results
    
    def _analyze_configuration_similarity(self, transfer_degradation: dict[str, t.Any]) -> dict[str, t.Any]:
        """Analyze how configuration similarity affects transfer performance."""
        similarity_analysis = {}
        
        # Extract all configuration pairs and their transfer performance
        config_pairs = []
        
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                
                # Compute various similarity metrics
                l_similarity = 1.0 / (1.0 + abs(source_L - target_L))
                m_similarity = 1.0 / (1.0 + abs(source_m - target_m))
                combined_similarity = (l_similarity + m_similarity) / 2.0
                
                config_pairs.append({
                    "source_config": (source_L, source_m),
                    "target_config": (target_L, target_m),
                    "l_similarity": l_similarity,
                    "m_similarity": m_similarity,
                    "combined_similarity": combined_similarity,
                    "degradation": degradation,
                    "transfer_condition": target_data["transfer_condition"]
                })
        
        if not config_pairs:
            return similarity_analysis
        
        pairs_df = pd.DataFrame(config_pairs)
        
        # Compute correlations between similarity and performance
        similarity_metrics = ["l_similarity", "m_similarity", "combined_similarity"]
        
        for metric in similarity_metrics:
            correlation, p_value = stats.spearmanr(pairs_df[metric], -pairs_df["degradation"])  # Negative because less degradation is better
            
            similarity_analysis[metric] = {
                "correlation": float(correlation),
                "p_value": float(p_value),
                "significant": p_value < 0.05
            }
        
        # Analyze by transfer condition
        condition_similarities = {}
        for condition in pairs_df["transfer_condition"].unique():
            condition_data = pairs_df[pairs_df["transfer_condition"] == condition]
            
            condition_correlations = {}
            for metric in similarity_metrics:
                correlation, p_value = stats.spearmanr(condition_data[metric], -condition_data["degradation"])
                condition_correlations[metric] = {
                    "correlation": float(correlation),
                    "p_value": float(p_value),
                    "significant": p_value < 0.05,
                    "n_samples": len(condition_data)
                }
            
            condition_similarities[condition] = condition_correlations
        
        similarity_analysis["by_condition"] = condition_similarities
        
        return similarity_analysis
    
    def _generate_transfer_visualizations(self, results: dict[str, t.Any]) -> None:
        """Generate comprehensive transfer analysis visualizations."""
        setup_publication_style()
        
        output_dir = self.output_dir / "visualizations"
        output_dir.mkdir(exist_ok=True)
        
        # 1. Transfer matrices heatmaps
        if "transfer_matrices" in results:
            self._plot_transfer_matrices(results["transfer_matrices"], output_dir)
        
        # 2. Degradation by transfer condition
        if "transfer_patterns" in results:
            self._plot_transfer_patterns(results["transfer_patterns"], output_dir)
        
        # 3. Configuration distance effects
        if "transfer_degradation" in results:
            self._plot_distance_effects(results["transfer_degradation"], output_dir)
        
        # 4. Significance analysis
        if "significance_analysis" in results:
            self._plot_significance_analysis(results["significance_analysis"], output_dir)
        
        # 5. Similarity analysis
        if "similarity_analysis" in results:
            self._plot_similarity_analysis(results["similarity_analysis"], output_dir)
    
    def _plot_transfer_matrices(self, transfer_matrices: dict[str, t.Any], output_dir: Path) -> None:
        """Plot transfer performance matrices."""
        for n_train_key, matrix_data in transfer_matrices.items():
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
            
            # Transfer performance matrix
            transfer_matrix = np.array(matrix_data["transfer_matrix"])
            labels = matrix_data["config_labels"]
            
            im1 = ax1.imshow(transfer_matrix, cmap=self.config.matrix_colormap, aspect="auto")
            ax1.set_xticks(range(len(labels)))
            ax1.set_yticks(range(len(labels)))
            ax1.set_xticklabels(labels, rotation=45, ha="right")
            ax1.set_yticklabels(labels)
            ax1.set_title(f"Transfer Performance Matrix ({n_train_key})")
            ax1.set_xlabel("Target Configuration")
            ax1.set_ylabel("Source Configuration")
            
            # Add colorbar
            cbar1 = plt.colorbar(im1, ax=ax1)
            cbar1.set_label("ICL Accuracy")
            
            # Degradation matrix
            degradation_matrix = np.array(matrix_data["degradation_matrix"])
            
            im2 = ax2.imshow(degradation_matrix, cmap=self.config.degradation_colormap, aspect="auto")
            ax2.set_xticks(range(len(labels)))
            ax2.set_yticks(range(len(labels)))
            ax2.set_xticklabels(labels, rotation=45, ha="right")
            ax2.set_yticklabels(labels)
            ax2.set_title(f"Transfer Degradation Matrix ({n_train_key})")
            ax2.set_xlabel("Target Configuration")
            ax2.set_ylabel("Source Configuration")
            
            # Add colorbar
            cbar2 = plt.colorbar(im2, ax=ax2)
            cbar2.set_label("Relative Performance Degradation")
            
            plt.tight_layout()
            save_figure(fig, output_dir / f"transfer_matrices_{n_train_key}.png")
            plt.close()
    
    def _plot_transfer_patterns(self, transfer_patterns: dict[str, t.Any], output_dir: Path) -> None:
        """Plot transfer degradation patterns by condition."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Degradation by transfer condition
        condition_data = transfer_patterns["condition_analysis"]
        if condition_data:
            conditions = list(condition_data.keys())
            means = [condition_data[cond]["mean_degradation"] for cond in conditions]
            stds = [condition_data[cond]["std_degradation"] for cond in conditions]
            
            bars = ax1.bar(conditions, means, yerr=stds, capsize=5, alpha=0.7)
            ax1.set_ylabel("Mean Degradation")
            ax1.set_title("Transfer Degradation by Condition")
            ax1.axhline(y=self.config.degradation_threshold, color='red', linestyle='--', 
                       label=f'Threshold ({self.config.degradation_threshold})')
            ax1.legend()
            
            # Add value labels on bars
            for bar, mean in zip(bars, means):
                height = bar.get_height()
                ax1.text(bar.get_x() + bar.get_width()/2., height + max(stds)/20,
                        f'{mean:.3f}', ha='center', va='bottom')
        
        # 2. Hierarchy (L) distance effects
        hierarchy_data = transfer_patterns["hierarchy_effects"]
        if hierarchy_data:
            distances = sorted([int(d) for d in hierarchy_data.keys()])
            means = [hierarchy_data[str(d)]["mean_degradation"] for d in distances]
            stds = [hierarchy_data[str(d)]["std_degradation"] for d in distances]
            
            ax2.errorbar(distances, means, yerr=stds, marker='o', capsize=5)
            ax2.set_xlabel("Hierarchy Distance (|L_source - L_target|)")
            ax2.set_ylabel("Mean Degradation")
            ax2.set_title("Degradation vs Hierarchy Distance")
            ax2.grid(True, alpha=0.3)
        
        # 3. Diversity (m) distance effects
        diversity_data = transfer_patterns["diversity_effects"]
        if diversity_data:
            distances = sorted([int(d) for d in diversity_data.keys()])
            means = [diversity_data[str(d)]["mean_degradation"] for d in distances]
            stds = [diversity_data[str(d)]["std_degradation"] for d in distances]
            
            ax3.errorbar(distances, means, yerr=stds, marker='s', capsize=5, color='orange')
            ax3.set_xlabel("Diversity Distance (|m_source - m_target|)")
            ax3.set_ylabel("Mean Degradation")
            ax3.set_title("Degradation vs Diversity Distance")
            ax3.grid(True, alpha=0.3)
        
        # 4. Total distance effects
        distance_data = transfer_patterns["distance_effects"]
        if distance_data:
            distances = sorted([int(d) for d in distance_data.keys()])
            means = [distance_data[str(d)]["mean_degradation"] for d in distances]
            stds = [distance_data[str(d)]["std_degradation"] for d in distances]
            
            ax4.errorbar(distances, means, yerr=stds, marker='^', capsize=5, color='green')
            ax4.set_xlabel("Total Distance (|L_source - L_target| + |m_source - m_target|)")
            ax4.set_ylabel("Mean Degradation")
            ax4.set_title("Degradation vs Total Configuration Distance")
            ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        save_figure(fig, output_dir / "transfer_patterns.png")
        plt.close()
    
    def _plot_distance_effects(self, transfer_degradation: dict[str, t.Any], output_dir: Path) -> None:
        """Plot detailed distance effects on transfer performance."""
        # Collect all transfer data
        all_transfers = []
        
        for source_key, source_data in transfer_degradation.items():
            source_L = source_data["source_config_L"]
            source_m = source_data["source_config_m"]
            source_n = source_data["source_n_train"]
            
            for target_key, target_data in source_data["transfer_results"].items():
                target_L = target_data["target_config_L"]
                target_m = target_data["target_config_m"]
                degradation = target_data["mean_degradation"]
                condition = target_data["transfer_condition"]
                
                l_distance = abs(source_L - target_L)
                m_distance = abs(source_m - target_m)
                
                all_transfers.append({
                    "l_distance": l_distance,
                    "m_distance": m_distance,
                    "total_distance": l_distance + m_distance,
                    "degradation": degradation,
                    "condition": condition,
                    "source_n_train": source_n
                })
        
        if not all_transfers:
            return
        
        df = pd.DataFrame(all_transfers)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Scatter plot: L distance vs degradation
        for condition in df["condition"].unique():
            condition_data = df[df["condition"] == condition]
            ax1.scatter(condition_data["l_distance"], condition_data["degradation"], 
                       label=condition, alpha=0.6)
        
        ax1.set_xlabel("Hierarchy Distance (L)")
        ax1.set_ylabel("Performance Degradation")
        ax1.set_title("Degradation vs Hierarchy Distance")
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. Scatter plot: m distance vs degradation
        for condition in df["condition"].unique():
            condition_data = df[df["condition"] == condition]
            ax2.scatter(condition_data["m_distance"], condition_data["degradation"], 
                       label=condition, alpha=0.6)
        
        ax2.set_xlabel("Diversity Distance (m)")
        ax2.set_ylabel("Performance Degradation")
        ax2.set_title("Degradation vs Diversity Distance")
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. Heatmap: L vs m distance
        pivot_data = df.groupby(["l_distance", "m_distance"])["degradation"].mean().unstack(fill_value=np.nan)
        
        if not pivot_data.empty:
            im = ax3.imshow(pivot_data.values, cmap="Reds", aspect="auto")
            ax3.set_xticks(range(len(pivot_data.columns)))
            ax3.set_yticks(range(len(pivot_data.index)))
            ax3.set_xticklabels(pivot_data.columns)
            ax3.set_yticklabels(pivot_data.index)
            ax3.set_xlabel("Diversity Distance (m)")
            ax3.set_ylabel("Hierarchy Distance (L)")
            ax3.set_title("Mean Degradation Heatmap")
            
            # Add colorbar
            cbar = plt.colorbar(im, ax=ax3)
            cbar.set_label("Mean Degradation")
        
        # 4. Box plot by n_train
        df.boxplot(column="degradation", by="source_n_train", ax=ax4)
        ax4.set_xlabel("Source Training Diversity (n_train)")
        ax4.set_ylabel("Performance Degradation")
        ax4.set_title("Degradation Distribution by Training Diversity")
        plt.suptitle("")  # Remove default title
        
        plt.tight_layout()
        save_figure(fig, output_dir / "distance_effects_detailed.png")
        plt.close()
    
    def _plot_significance_analysis(self, significance_analysis: dict[str, t.Any], output_dir: Path) -> None:
        """Plot statistical significance analysis results."""
        # Collect significance data
        all_tests = []
        
        for source_key, source_data in significance_analysis.items():
            for target_key, test_data in source_data["transfer_tests"].items():
                all_tests.append({
                    "source_config": f"L{source_data['source_config_L']}_m{source_data['source_config_m']}",
                    "target_config": f"L{test_data['target_config_L']}_m{test_data['target_config_M']}",
                    "condition": test_data["transfer_condition"],
                    "p_value": test_data["p_value"],
                    "p_corrected": test_data["p_corrected"],
                    "effect_size": test_data["effect_size"],
                    "significant": test_data["significant"],
                    "baseline_mean": test_data["baseline_mean"],
                    "transfer_mean": test_data["transfer_mean"]
                })
        
        if not all_tests:
            return
        
        df = pd.DataFrame(all_tests)
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. P-value distribution
        ax1.hist(df["p_value"], bins=20, alpha=0.7, edgecolor='black')
        ax1.axvline(x=0.05, color='red', linestyle='--', label='α = 0.05')
        ax1.set_xlabel("P-value")
        ax1.set_ylabel("Frequency")
        ax1.set_title("Distribution of P-values")
        ax1.legend()
        
        # 2. Effect size distribution
        ax2.hist(df["effect_size"], bins=20, alpha=0.7, edgecolor='black', color='orange')
        ax2.set_xlabel("Effect Size (Cohen's d)")
        ax2.set_ylabel("Frequency")
        ax2.set_title("Distribution of Effect Sizes")
        ax2.axvline(x=0, color='black', linestyle='-', alpha=0.5)
        
        # 3. Significance by condition
        condition_counts = df.groupby(["condition", "significant"]).size().unstack(fill_value=0)
        
        if not condition_counts.empty:
            condition_counts.plot(kind="bar", ax=ax3, stacked=True)
            ax3.set_xlabel("Transfer Condition")
            ax3.set_ylabel("Number of Tests")
            ax3.set_title("Significance Results by Condition")
            ax3.legend(["Not Significant", "Significant"])
            ax3.tick_params(axis='x', rotation=45)
        
        # 4. Effect size vs p-value
        colors = ['red' if sig else 'blue' for sig in df["significant"]]
        ax4.scatter(df["effect_size"], -np.log10(df["p_value"]), c=colors, alpha=0.6)
        ax4.axhline(y=-np.log10(0.05), color='red', linestyle='--', label='α = 0.05')
        ax4.set_xlabel("Effect Size")
        ax4.set_ylabel("-log10(p-value)")
        ax4.set_title("Volcano Plot: Effect Size vs Significance")
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        save_figure(fig, output_dir / "significance_analysis.png")
        plt.close()
    
    def _plot_similarity_analysis(self, similarity_analysis: dict[str, t.Any], output_dir: Path) -> None:
        """Plot configuration similarity analysis results."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Overall similarity correlations
        metrics = ["l_similarity", "m_similarity", "combined_similarity"]
        correlations = [similarity_analysis[metric]["correlation"] for metric in metrics]
        p_values = [similarity_analysis[metric]["p_value"] for metric in metrics]
        
        bars = ax1.bar(metrics, correlations, 
                      color=['red' if p < 0.05 else 'gray' for p in p_values])
        ax1.set_ylabel("Correlation with Transfer Success")
        ax1.set_title("Similarity Metrics vs Transfer Performance")
        ax1.tick_params(axis='x', rotation=45)
        
        # Add significance indicators
        for i, (bar, p_val) in enumerate(zip(bars, p_values)):
            height = bar.get_height()
            if p_val < 0.05:
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'p={p_val:.3f}*', ha='center', va='bottom')
            else:
                ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'p={p_val:.3f}', ha='center', va='bottom')
        
        # 2. Correlation by transfer condition
        if "by_condition" in similarity_analysis:
            condition_data = similarity_analysis["by_condition"]
            conditions = list(condition_data.keys())
            
            for i, metric in enumerate(["l_similarity", "m_similarity", "combined_similarity"]):
                metric_corrs = [condition_data[cond][metric]["correlation"] for cond in conditions]
                metric_ps = [condition_data[cond][metric]["p_value"] for cond in conditions]
                
                x_pos = np.arange(len(conditions)) + i * 0.25
                bars = ax2.bar(x_pos, metric_corrs, width=0.25, label=metric.replace('_', ' ').title(),
                              alpha=0.7)
                
                # Add significance markers
                for bar, p_val in zip(bars, metric_ps):
                    if p_val < 0.05:
                        height = bar.get_height()
                        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                                '*', ha='center', va='bottom', fontweight='bold')
            
            ax2.set_xlabel("Transfer Condition")
            ax2.set_ylabel("Correlation")
            ax2.set_title("Similarity Correlations by Condition")
            ax2.set_xticks(np.arange(len(conditions)) + 0.25)
            ax2.set_xticklabels(conditions)
            ax2.legend()
        
        # 3. & 4. Placeholder for additional similarity analyses
        ax3.text(0.5, 0.5, "Additional similarity\nanalysis plots\ncan be added here",
                ha='center', va='center', transform=ax3.transAxes, fontsize=12)
        ax3.set_title("Future Similarity Analysis")
        
        ax4.text(0.5, 0.5, "Configuration space\nvisualization\ncan be added here",
                ha='center', va='center', transform=ax4.transAxes, fontsize=12)
        ax4.set_title("Configuration Space Visualization")
        
        plt.tight_layout()
        save_figure(fig, output_dir / "similarity_analysis.png")
        plt.close()


def run_rq4_analysis(
    data_dir: Path,
    output_dir: Path,
    config: TransferConfig | None = None
) -> dict[str, t.Any]:
    """Run RQ4 transfer analysis with default configuration."""
    if config is None:
        config = TransferConfig(
            data_dir=data_dir,
            output_dir=output_dir,
            transfer_conditions=["cross_L", "cross_m", "cross_config"],
            min_samples_for_transfer=5,
            degradation_threshold=0.1,
            compute_transfer_matrices=True,
            matrix_aggregation="mean",
            transfer_significance_test="ttest",
            multiple_comparisons_correction="bonferroni"
        )
    
    analyzer = TransferAnalyzer(config)
    return analyzer.run_analysis()


# Test function
def test_transfer_analyzer():
    """Test the transfer analyzer with sample data."""
    # This would typically use real data paths
    data_dir = Path("./test_data")
    output_dir = Path("./test_output/rq4")
    
    # Create test configuration
    config = TransferConfig(
        data_dir=data_dir,
        output_dir=output_dir,
        transfer_conditions=["cross_L", "cross_m"],
        min_samples_for_transfer=3,
        degradation_threshold=0.05,
        matrix_aggregation="mean"
    )
    
    try:
        results = run_rq4_analysis(data_dir, output_dir, config)
        print("✓ RQ4 Transfer analysis completed successfully")
        print(f"✓ Results saved to {output_dir}")
        return results
    except Exception as e:
        print(f"✗ Transfer analysis failed: {e}")
        return None


if __name__ == "__main__":
    test_transfer_analyzer()

# Q5 Diversity analysis

In [ ]:
"""RQ5: Analyze task diversity effects on out-of-distribution performance."""

from pathlib import Path
from dataclasses import dataclass, field
import typing as t
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from .base_analyzer import BaseAnalyzer, AnalysisConfig
from ..utils.data_loaders import create_filter
from ..utils.statistical_utils import compute_confidence_interval, bootstrap_test, fit_power_law
from ..utils.visualization_utils import setup_publication_style, save_figure


@dataclass
class DiversityConfig(AnalysisConfig):
    """Configuration for diversity analysis."""
    # Diversity analysis parameters
    min_n_train_samples: int = 3
    max_n_train_for_analysis: int | None = None
    
    # OOD vs ID comparison
    ood_transfer_conditions: list[str] = field(default_factory=lambda: [
        "cross_L", "cross_m", "cross_config"
    ])
    id_transfer_condition: str = "within_config"
    
    # Memorization vs generalization analysis
    analyze_memorization: bool = True
    context_size_thresholds: list[int] = field(default_factory=lambda: [1, 5, 10, 20])
    
    # Scaling law fitting
    fit_scaling_laws: bool = True
    scaling_law_forms: list[str] = field(default_factory=lambda: [
        "power", "exponential", "logarithmic"
    ])
    
    # Statistical analysis
    diversity_significance_test: str = "anova"  # "anova", "kruskal"
    correlation_method: str = "spearman"  # "pearson", "spearman"
    
    # Visualization parameters
    diversity_colormap: str = "viridis"
    performance_colormap: str = "RdYlBu_r"


class DiversityAnalyzer(BaseAnalyzer):
    """Analyzer for task diversity effects on OOD performance (RQ5)."""
    
    def __init__(self, config: DiversityConfig):
        """Initialize diversity analyzer."""
        super().__init__(config)
        self.config: DiversityConfig = config
    
    def run_analysis(self) -> dict[str, t.Any]:
        """Run comprehensive diversity analysis."""
        print("Starting RQ5: Diversity Analysis")
        print("=" * 50)
        
        # Load and validate data
        id_data = self._load_id_performance_data()
        ood_data = self._load_ood_performance_data()
        
        if id_data.empty:
            raise ValueError("Insufficient in-distribution data for diversity analysis")
        
        print(f"Loaded {len(id_data)} in-distribution evaluation records")
        print(f"Loaded {len(ood_data)} out-of-distribution evaluation records")
        print(f"n_train levels: {sorted(id_data['n_train'].unique())}")
        
        results = {}
        
        # 1. Analyze diversity scaling on ID performance
        print("\n1. Analyzing diversity scaling on ID performance...")
        id_scaling = self._analyze_id_diversity_scaling(id_data)
        results["id_diversity_scaling"] = id_scaling
        
        # 2. Compare ID vs OOD performance scaling
        print("2. Comparing ID vs OOD performance scaling...")
        if not ood_data.empty:
            ood_comparison = self._compare_id_ood_scaling(id_data, ood_data)
            results["id_ood_comparison"] = ood_comparison
        
        # 3. Analyze memorization vs generalization trade-offs
        print("3. Analyzing memorization vs generalization...")
        if self.config.analyze_memorization:
            memorization_analysis = self._analyze_memorization_generalization(id_data, ood_data)
            results["memorization_analysis"] = memorization_analysis
        
        # 4. Fit diversity scaling laws
        print("4. Fitting diversity scaling laws...")
        if self.config.fit_scaling_laws:
            scaling_laws = self._fit_diversity_scaling_laws(id_data, ood_data)
            results["scaling_laws"] = scaling_laws
        
        # 5. Analyze diversity-context interactions
        print("5. Analyzing diversity-context interactions...")
        interaction_analysis = self._analyze_diversity_context_interactions(id_data, ood_data)
        results["diversity_context_interactions"] = interaction_analysis
        
        # 6. Statistical significance testing
        print("6. Testing diversity effects significance...")
        significance_analysis = self._test_diversity_significance(id_data, ood_data)
        results["significance_analysis"] = significance_analysis
        
        # 7. Generate visualizations
        print("7. Generating visualizations...")
        self._generate_diversity_visualizations(results)
        
        # Save results
        self.save_results(results, "rq5_diversity_results.json")
        
        print("\nRQ5 Analysis completed successfully!")
        return results
    
    def _load_id_performance_data(self) -> pd.DataFrame:
        """Load in-distribution performance data."""
        filter_dict = (create_filter()
                      .transfer_condition(self.config.id_transfer_condition)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by n_train if specified
        if self.config.max_n_train_for_analysis is not None:
            data = data[data["n_train"] <= self.config.max_n_train_for_analysis]
        
        return data
    
    def _load_ood_performance_data(self) -> pd.DataFrame:
        """Load out-of-distribution performance data."""
        filter_dict = (create_filter()
                      .transfer_condition(self.config.ood_transfer_conditions)
                      .control_type("normal")
                      .build())
        
        data = self.data_loader.load_icl_performance(filters=filter_dict)
        
        # Add model metadata
        data = data.merge(
            self.model_registry[["model_id", "config_L", "config_m", "n_train", "checkpoint_step"]],
            on="model_id",
            how="left"
        )
        
        # Filter by n_train if specified
        if self.config.max_n_train_for_analysis is not None:
            data = data[data["n_train"] <= self.config.max_n_train_for_analysis]
        
        return data
    
    def _analyze_id_diversity_scaling(self, id_data: pd.DataFrame) -> dict[str, t.Any]:
        """Analyze how task diversity affects in-distribution performance."""
        scaling_results = {}
        
        # Group by configuration and analyze diversity scaling
        for (config_L, config_m), config_group in id_data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Get performance vs diversity data
            diversity_performance = []
            
            for n_train in sorted(config_group["n_train"].unique()):
                n_train_data = config_group[config_group["n_train"] == n_train]
                
                if len(n_train_data) < self.config.min_n_train_samples:
                    continue
                
                # Aggregate across context sizes and models
                performance_stats = {
                    "n_train": n_train,
                    "mean_accuracy": float(n_train_data["accuracy"].mean()),
                    "std_accuracy": float(n_train_data["accuracy"].std()),
                    "median_accuracy": float(n_train_data["accuracy"].median()),
                    "max_accuracy": float(n_train_data["accuracy"].max()),
                    "min_accuracy": float(n_train_data["accuracy"].min()),
                    "n_samples": len(n_train_data),
                    "n_models": n_train_data["model_id"].nunique(),
                    "n_contexts": n_train_data["context_size"].nunique()
                }
                
                # Compute confidence intervals
                ci = compute_confidence_interval(n_train_data["accuracy"].values)
                performance_stats["ci_lower"] = float(ci[0])
                performance_stats["ci_upper"] = float(ci[1])
                
                # Analyze by context size
                context_performance = {}
                for context_size in sorted(n_train_data["context_size"].unique()):
                    context_data = n_train_data[n_train_data["context_size"] == context_size]
                    context_performance[int(context_size)] = {
                        "mean_accuracy": float(context_data["accuracy"].mean()),
                        "std_accuracy": float(context_data["accuracy"].std()),
                        "n_samples": len(context_data)
                    }
                
                performance_stats["by_context"] = context_performance
                diversity_performance.append(performance_stats)
            
            if len(diversity_performance) >= 2:
                # Compute scaling trends
                n_trains = [p["n_train"] for p in diversity_performance]
                accuracies = [p["mean_accuracy"] for p in diversity_performance]
                
                # Linear correlation
                correlation, p_value = stats.spearmanr(n_trains, accuracies)
                
                # Fit trends
                trends = self._fit_diversity_trends(n_trains, accuracies)
                
                scaling_results[config_key] = {
                    "config_L": config_L,
                    "config_m": config_m,
                    "diversity_performance": diversity_performance,
                    "correlation": float(correlation),
                    "correlation_p_value": float(p_value),
                    "trends": trends,
                    "n_diversity_levels": len(diversity_performance)
                }
        
        return scaling_results
    
    def _compare_id_ood_scaling(self, id_data: pd.DataFrame, ood_data: pd.DataFrame) -> dict[str, t.Any]:
        """Compare in-distribution vs out-of-distribution diversity scaling."""
        comparison_results = {}
        
        # Group by source configuration
        for (config_L, config_m), id_group in id_data.groupby(["config_L", "config_m"]):
            config_key = f"L{config_L}_m{config_m}"
            
            # Get ID scaling
            id_scaling = []
            for n_train in sorted(id_group["n_train"].unique()):
                n_train_data = id_group[id_group["n_train"] == n_train]
                if len(n_train_data) >= self.config.min_n_train_samples:
                    id_scaling.append({
                        "n_train": n_train,
                        "mean_accuracy": float(n_train_data["accuracy"].mean()),
                        "std_accuracy": float(n_train_data["accuracy"].std())
                    })
            
            # Get OOD scaling for each transfer condition
            ood_scaling_by_condition = {}
            
            ood_config_data = ood_data[
                (ood_data["config_L"] == config_L) &
                (ood_data["config_m"] == config_m)
            ]
            
            for transfer_condition in self.config.ood_transfer_conditions:
                condition_data = ood_config_data[
                    ood_config_data["transfer_condition"] == transfer_condition
                ]
                
                if condition_data.empty:
                    continue

# Q6 Comparative Analysis